In [ ]:
# Setup: in Google Colab, download and install the repository (a few minutes, once per session); elsewhere, work from the
# repository. Pages that read the derived catalogues need Zenodo bundle 1 (python scripts/fetch_release.py).
import os, subprocess, sys
os.environ['MPLBACKEND'] = 'Agg'   # as in a plain script run: Jupyter's inline settings would change the saved figures
if 'google.colab' in sys.modules:
    if not os.path.exists('/content/lmc-ancient-disc'):
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/vasilybelokurov/lmc-ancient-disc', '/content/lmc-ancient-disc'], check=True)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '/content/lmc-ancient-disc'], check=True)
    os.chdir('/content/lmc-ancient-disc')
else:
    while not os.path.exists('pyproject.toml'):
        os.chdir('..')
sys.path[:0] = ['src', 'scripts', 'kb']
if not os.path.exists('data/giants_apogee.parquet'):   # the catalogues of Zenodo bundle 1, checked against data/sources.yaml
    subprocess.run([sys.executable, 'scripts/fetch_release.py'], check=True)

# Stellar mass per metallicity

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/lmc-ancient-disc/blob/main/kb/notebooks/chapters/mass.ipynb)

Every number on this page is written by `scripts/paper_text_mass.py` from the result tables (`results/macros_mass.tex`); the code
cells are the figure scripts themselves, step by step, and the tables are read from the result files.

The Letter compares the LMC's old stars with simulated galaxies at the stellar mass the LMC had formed when those stars were born.
That comparison needs the stellar mass the LMC formed at each metallicity. Metallicity here is the iron abundance [Fe/H], the
logarithm of the ratio of iron to hydrogen atoms relative to the Sun's (so [Fe/H] = −1 is one tenth of the solar iron abundance).
This page converts the red giants with spectra into that mass. The giants come from two spectroscopic surveys:
[APOGEE DR17](https://doi.org/10.3847/1538-4365/ac4414) (the seventeenth data release of the Sloan Digital Sky Survey, SDSS, with
near-infrared spectra analysed by its pipeline ASPCAP) and the BOSS optical spectra of [SDSS-V DR20](https://doi.org/10.48550/arXiv.2607.26149),
whose stellar parameters come from the neural network BOSSNet ([Sizemore et al. 2024](https://doi.org/10.3847/1538-3881/ad291d)). Both are
placed on one scale: the BOSS [Fe/H] is calibrated to APOGEE on the stars observed by both (chapter 3). Positions are angular
distances $\rho$ from the LMC centre of [van der Marel & Kallivayalil (2014)](https://doi.org/10.1088/0004-637X/781/2/121); magnitudes
$G$ and colours $G_{\rm BP}-G_{\rm RP}$ are from [*Gaia* DR3](https://doi.org/10.1051/0004-6361/202243940), $H$ from
[2MASS](https://doi.org/10.1086/498708). M☉ is the mass of the Sun.

**Mass samples.** The masses use the quality cuts of the selection functions (chapter 2), without the kinematic cuts of the samples
used for the velocities, with the giant cut and 150 < $v_{\rm los}$ < 420 km/s, where
$v_{\rm los}$ is the line-of-sight velocity (calibrated velocities for BOSS). They hold 57 557 BOSS and
5 796 APOGEE giants with positive selection weights.

We convert the weighted BOSS giants into the stellar mass formed in each [Fe/H] bin within $\rho$ = 12°. Bins are
0.25 dex wide from -2 to 0; the first collects all stars below
-2 and the last all stars above 0. Three steps lead from the spectra to the mass: corrected
counts, giants per unit mass from stellar-population models, and the response of the measured BOSS [Fe/H] to the true one.

## Corrected counts

The corrected number of LMC giants in bin $k$ is

$$
N_k=\sum_{i\in k}w_i\,c_i\,\bigl(1-f_{{\rm MW},i}\bigr),
$$

where the sum runs over the BOSS giants $i$ with [Fe/H] in bin $k$. The selection weight $w_i$ is the number of stars of the BOSS
target list (the parent sample, rebuilt from *Gaia* DR3 as the SDSS-V target class, or carton, `mwm_magcloud_rgb_boss`, selected it) that
each giant with a good spectrum stands for; the coverage factor $c_i$ adds the parent stars in sky areas where too few spectra were
taken to compute a weight. Both are derived in chapter 2. $f_{{\rm MW},i}$ is the fraction of Milky Way foreground stars expected at the
position and metallicity of star $i$. It comes from a fit of the LMC disc and a Milky Way component to the velocities and proper
motions of the giants per [Fe/H] group (Figure 4.7), which gives mean fractions of
6–9 per cent below [Fe/H] = −1.5 and about 1 per cent above −1
over $\rho$ = 2–10°. We assume a foreground of uniform surface density in each [Fe/H] bin, so
$f_{\rm MW}(\rho)$ is this mean fraction times the ratio of the mean surface density of the bin over
2–10° to its surface density at $\rho$. The metal-poor giants whose proper motions lie more than
0.8 mas/yr from the local LMC motion, a sample dominated by the foreground, have a surface density that is constant
with radius within its errors ($\chi^2$ = 6.8 for 3 degrees of freedom, set by
18 stars at 6–9°; see the checks below).

## Giants per unit mass

A stellar population of formed mass $M$, age $\tau$ and metallicity [M/H] places $\eta([{\rm M/H}],\tau)\,M$ stars inside the BOSS
selection. [M/H] is the abundance of all metals relative to the Sun's, which the stellar models take as input; $\eta$, the number of
selected giants per solar mass formed, is the efficiency of the conversion. We compute $\eta$ from the PARSEC v1.2S stellar isochrones
([Bressan et al. 2012](https://doi.org/10.1111/j.1365-2966.2012.21948.x)) with the COLIBRI tracks of the thermally pulsing asymptotic giant
branch (TP-AGB) calibrated on the LMC ([Marigo et al. 2017](https://doi.org/10.3847/1538-4357/835/1/77);
[Pastorelli et al. 2020](https://doi.org/10.1093/mnras/staa2565)), for a [Kroupa (2001)](https://doi.org/10.1046/j.1365-8711.2001.04022.x)
initial mass function (IMF) normalised to unit mass between 0.08 and 100 M☉. Each isochrone is placed
at the distance modulus 18.477 ([Pietrzyński et al. 2019](https://doi.org/10.1038/s41586-019-0999-4)) plus the offset of
the inclined disc (inclination 34°, line of nodes at position angle 139.1°;
[van der Marel & Kallivayalil 2014](https://doi.org/10.1088/0004-637X/781/2/121)), smeared with a Gaussian depth of
0.1 mag, and reddened with the true extinction. Reddening is measured by the colour excess $E(B-V)$, the extra
$B-V$ colour that dust adds. Inside the footprint of the OGLE-IV survey ([Udalski et al. 2015](https://ui.adsabs.harvard.edu/abs/2015AcA....65....1U))
we take the colour excess $E(V-I)$ from the red-clump map of [Skowron et al. (2021)](https://doi.org/10.3847/1538-4365/abcb81) and convert
it with $E(V-I)/E(B-V)$ = 1.237 ([Schlafly & Finkbeiner 2011](https://doi.org/10.1088/0004-637X/737/2/103), Table 6);
outside it we use the dust map of [Schlegel, Finkbeiner & Davis (1998)](https://doi.org/10.1086/305772) (SFD). The carton's polygon in
the colour–magnitude diagram (CMD) is then applied to the photometry as the carton dereddened it, with its own reddening, and the
limit $G\leq$ 17.5 to the observed magnitude. The difference matters: the median true $E(B-V)$ of the parent is
0.083, while the carton removed 0.038. We count the isochrone stars inside the selection by
integrating the cumulative IMF along each isochrone, in 40 steps between tabulated points. The calculation runs
separately in 16 regions (rings with edges at 0, 3, 6, 9 and 12°, each split into quadrants), at the median
extinction and disc offset of their parent stars; the offsets span -0.19 to +0.28 mag. Applying the
regional median to dusty patches underestimates the stars they lose; an infrared test (Figure 4.6) puts the
missing parent stars at 1.1 per cent of the central 4.5°.

The models need [M/H], an age and a mass for each [Fe/H] bin. We convert with
[M/H] = [Fe/H] + log(0.638 × 10<sup>[α/Fe]</sup> + 0.362)
([Salaris, Chieffi & Straniero 1993](https://doi.org/10.1086/173105)), where [α/Fe] is the abundance of the α elements (O, Mg, Si,
Ca, Ti) relative to iron, in solar units. For [α/Fe] we use the median ASPCAP [α/M] of the APOGEE members in the bin:
0.25 at [Fe/H] = -2.1, 0.14 at -1.6, 0.06 at
-1.1 and 0.02 at -0.4, held constant beyond the measured range. The giants of
one [Fe/H] bin span a range of ages, so we average $\eta$ over the distribution of formed mass in age expected for the bin. We take it
from the age–metallicity relation of the LMC field giants of [Povick et al. (2024)](https://doi.org/10.1093/mnras/stae1949): a median
[Fe/H] falling from -0.40 at the present to -0.95 at 12.5 Gyr, with a Gaussian scatter
of 0.15–0.24 dex (read from their fig. 20). With a constant star-formation rate, a bin's ages
are weighted by the probability that a star of each age has the bin's [Fe/H]. Table 4.1 gives the resulting ages for four [Fe/H]
ranges, together with three fixed windows of uniform formed mass that we keep as alternatives: A, which changes the masses below
[Fe/H] = −2, −1.5 and −1 and the total by -6 to 0 per cent from the reference, and B and C,
which shift the windows older and younger. The isochrone grid ends at 12.59 Gyr, so older ages (windows A and B) are
truncated there, and the grid starts at [M/H] = -2.19. Figure 4.4(a) shows $\eta$ for four
metallicities: between the grid ages nearest 2 and 12 Gyr it falls by a factor of
2.7 at [M/H] = -2 and 4.6 at [M/H] = -0.2, the main
reason why the age prior matters.

In [ ]:
import json
import pandas as pd
from lmcold.mass_conversion import AGE_RANGES
# Reference ages [Gyr] per [Fe/H] bin (16th-84th percentiles and mean) from the age-metallicity relation
a = json.load(open('results/age_weighting.json'))
rows = {'Reference (age–metallicity relation)': [f"{v['p16_p84'][0]:.1f}–{v['p16_p84'][1]:.1f} ({v['mean']:.1f})" for v in a.values()]}
# The age windows [Gyr] of lmcold.mass_conversion, one row each
rows.update({f'Window {k}' + {'A': '', 'B': ' (older)', 'C': ' (younger)'}[k]: [f'{lo:g}–{hi:g}' for lo, hi in w]
             for k, w in AGE_RANGES.items()})
pd.DataFrame(rows, index=['[Fe/H] < −1.5', '−1.5 to −1', '−1 to −0.5', '> −0.5']).T

**Table 4.1.** Ages (Gyr) of the formed mass per [Fe/H] range: the reference weighting from the age–metallicity relation of
Povick et al. (2024), 16th–84th percentiles with the mean in parentheses (`results/age_weighting.json`, written by
`scripts/paper_mass_method.py`), and the windows of uniform formed mass of the alternatives A, B and C
(`lmcold.mass_conversion.AGE_RANGES`).

## The BOSS metallicity response

The mass at low [Fe/H] is sensitive to how BOSS measures metallicity. On the 1 923 stars of the two mass samples
observed by both surveys, calibrated BOSS agrees with APOGEE to within 0.05 dex in median above
[Fe/H]<sub>APOGEE</sub> = −1.5 and below −2, but is 0.15–0.17 dex too low at
−2 < [Fe/H]<sub>APOGEE</sub> < −1.5 (Figure 4.1). Stars of this range are therefore measured below −2: in the parent
sample common to both surveys, BOSS finds 1.9 times as many stars below −2 as APOGEE
(Figure 4.4b, grey). A model of independent Gaussian errors does not describe BOSSNet, whose estimates are drawn towards
the bulk of its training set, so we measure the response directly. The response matrix $R_{jk}$ is the fraction of stars with APOGEE
[Fe/H] in bin $k$ that BOSS places in bin $j$, built from the stars observed by both and weighted to the joint distribution of $G$
(edges 16.25 and 16.75) and BOSS signal-to-noise ratio S/N (edge 30) of the sample being corrected,
separately above and below [Fe/H] = -1.5: the scatter of BOSS [Fe/H] grows as S/N falls, and the stars observed
by both are brighter and have higher S/N than the BOSS sample (median S/N 30 against 20
for the BOSS giants at $G$ = 16.75–17.5). With $\eta_k$ the efficiency and $M_k$ the
formed mass of true bin $k$, the expected BOSS counts in measured bin $j$ are then

$$
\langle N_j\rangle=\sum_k R_{jk}\,\eta_k\,M_k .
$$

Passed through $R$, the APOGEE counts of the common parent reproduce the BOSS counts ($\chi^2$ = 6.6 for
9 bins with the bootstrap covariance); below [Fe/H] = −2 BOSS holds 0.84
(+0.36/−0.28) of the prediction, against 1.87 without the response
(Figure 4.4b, blue).

We fit the masses $M_k\geq0$ of this equation to the BOSS counts in 0.125-dex measured bins by maximising a Poisson
likelihood, in true bins below −2, from −2 to −1.5, from −1.5 to −1 and above −1. Narrower true bins oscillate: the response rests on
1 923 stars, of which 78 lie below −1.5. The fit changes the mass of these four ranges by factors
of 0.54, 0.85, 0.87 and 1.01. Within each range below −1, the
mass is shared among its 0.25-dex bins as the uncorrected masses are.

Above −1 the BOSS values cannot share out the mass. BOSSNet overestimates the metallicity of hot giants: on the stars observed by
both surveys with APOGEE [Fe/H] between −1 and −0.5, BOSS reads +0.05–0.06 dex high at
effective temperature $T_{\rm eff}$ = 4300–4500 K and +0.14 to
+0.17 dex at 4500–4700 K, against at most 0.03 dex below
4300 K (chapter 3). Hot, low-S/N giants (above 4300 K and below S/N = 25) carry
46 per cent of the BOSS weight but are 5 per cent of the stars that define the calibration
and the response, so neither corrects them, and BOSS places 48 per cent of them above [Fe/H] = −0.5. We therefore
keep the BOSS total above −1, which agrees with APOGEE in the field cores (Figure 4.5), and share it among the bins above
−1 as the APOGEE giants of the same cell: each BOSS giant above −1 is distributed as the weighted [Fe/H] of the APOGEE giants above −1
with the same ring (0–3, 3–6, 6–12°), $G$ (edges 16.25, 16.75 and 17.1) and dereddened $G_{\rm BP}-G_{\rm RP}$ (edges
1.2, 1.4 and 1.6), or of the next coarser cell with at least 30 APOGEE giants, and converted with the
efficiency of its own region and bin.

<details><summary>About <code>scripts/boss_response.py</code></summary>

```text
BOSS metallicity response on the APOGEE scale, measured on the stars with both spectra, and its closure test (§2.3).

Overlap: the mass samples of both surveys (lmcold.mass_samples), same Gaia source; APOGEE [Fe/H] is the reference (true) scale.
The calibrated BOSS values are biased by -0.15 to -0.2 dex at -2 < [Fe/H] < -1.5 and unbiased on either side, and BOSSNet errors
shrink towards the bulk of its training set, so the response is measured directly (lmcold.mass_response.empirical_response),
not modelled as independent Gaussian errors (that model predicts a broadened MDF peak, which BOSS does not show).
G: the overlap is brighter than the BOSS sample, so its stars are weighted to the G distribution of the target sample
(lmcold.mass_samples.boss_response).
Closure: common parent (Gaia IDs in covered BOSS and APOGEE parents, as in paper_giant_counts). Predicted BOSS counts
R @ (APOGEE counts in the true bins) against the observed BOSS counts, in the reporting bins and below -2 and -1.5; errors
from 500 joint bootstraps of nside-16 pixels (both surveys, weights fixed) and overlap stars. Also by APOGEE field type
(LMC1-17 used the Washington + DDO51 giant flag, which is not reproduced; apogee_selection T7) and with the response measured
on the overlap stars of one half of the sky (nside-8 pixel parity) and tested on the other. The overlap stars are themselves
part of the common parent, so the full-sky closure is not independent of the response; the split-sky test is.
Writes results/boss_response.json and figures/paper_boss_response.png (appendix: overlap residuals; closure by field type).
The common-parent closure is panel (b) of figures/paper_mass_method.png.

Inputs (through lmcold.mass_samples.samples): data/boss_weights.parquet (release_boss_weights), data/apogee_weights.parquet
(release_apogee_weights), data/spec_giants.parquet and results/boss_calibration.json (scripts/build_giants.py); here also
apogee_weights.parquet for the APOGEE field of each common-parent star. Both samples: good giant spectra with positive weight
and 150 < v_los < 420 km/s (APOGEE also quality and covered).
Response matrix R: column j = distribution of calibrated BOSS [Fe/H] over the 0.125-dex measured bins (MEASURED_EDGES) for
overlap stars with APOGEE [Fe/H] in true bin j (TRUE_EDGES: < -2, -2..-1.5, -1.5..-1, > -1). Predicted BOSS counts
= R @ (weighted APOGEE counts per true bin), summed into the reporting bins (REPORT_EDGES); chi2 uses the bootstrap covariance
of observed minus predicted. results/boss_response.json: overlap size, median BOSS - APOGEE per reporting bin, the bin edges
and, per closure case, chi2, dof, observed / predicted / APOGEE counts per reporting bin, the 16th and 84th percentiles of the
bootstrap ratio, and the totals below -2 and -1.5.

Run: .venv/bin/python scripts/boss_response.py   (after boss_selection.py, apogee_selection.py, build_giants.py)
```
</details>

**Data read by `scripts/boss_response.py`** (from the run log)

| File | Where it comes from | What it is |
|---|---|---|
| `data/apogee_weights.parquet` | Zenodo bundle 1 | APOGEE selection weights per candidate star (selection groups, coverage, w = N_cand / N_quality x pass factor in H x XP [M/H]) |
| `data/boss_weights.parquet` | Zenodo bundle 1 | BOSS selection weights per eligible parent star (selection groups, w = N_elig / N_good x pass factor in G x XP [M/H] x ring) |
| `data/spec_giants.parquet` | Zenodo bundle 1 | APOGEE DR17 and SDSS-V DR20 BOSS spectra of LMC giant candidates before the Gaia match |
| `results/boss_calibration.json` | repository | calibration of the BOSS [Fe/H] and velocities to APOGEE on the stars observed by both; written by `python scripts/build_giants.py` |

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

In [ ]:
import json

import healpy as hp
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from lmcold.layout import panel_grid
from lmcold.mass_samples import samples, boss_response, overlap, FEH_CENTRES, FEH_EDGES, TRUE_EDGES, MEASURED_EDGES
from lmcold.sources import DATA, ROOT

### Settings and helpers
Number of joint bootstrap draws, the reporting [Fe/H] bins, and two helpers: the response matrix for a target sample
(`lmcold.mass_samples.boss_response`) and the sum of the fine measured bins into the reporting bins.

In [ ]:
NBOOT = 500  # joint bootstrap draws
# Reporting bins [dex]: FEH_EDGES with the open ends closed at -3 and +0.5 (centres FEH_CENTRES).
REPORT_EDGES = [-3., *FEH_EDGES[1:-1], .5]
# Reporting bin of each measured bin, by its upper edge; the last reporting bin also takes any measured bin beyond it.
GROUP = np.minimum(np.searchsorted(REPORT_EDGES, MEASURED_EDGES[1:]) - 1, len(FEH_CENTRES) - 1)  # reporting bin per measured bin


def response(o, t, w):
    """BOSS response matrix for target sample t, measured on overlap stars o.

    Parameters
    ----------
    o : pandas.DataFrame
        Overlap stars (lmcold.mass_samples.overlap: feh_b, feh_a, g, snr).
    t : pandas.DataFrame
        Target BOSS stars (feh [dex, calibrated], g [mag], snr).
    w : numpy.ndarray
        Selection weights of t.

    Returns
    -------
    numpy.ndarray
        Shape (len(MEASURED_EDGES) - 1, len(TRUE_EDGES) - 1); each column sums to 1.
    """
    # lmcold.mass_samples.boss_response weights the overlap stars to the G distribution of the target (module docstring).
    # Arguments: overlap stars, target [Fe/H] [dex], G [mag], S/N, and weights.
    return boss_response(o, t.feh, t.g, t.snr, w)


def coarse(x):
    """Measured-bin counts summed into the reporting bins (last axis).

    Parameters
    ----------
    x : numpy.ndarray
        Counts with the measured bins on the last axis.

    Returns
    -------
    numpy.ndarray
        Same leading shape, last axis len(FEH_CENTRES).
    """
    # For each reporting bin i, sum the measured bins whose GROUP is i; stack them on a new last axis.
    return np.stack([x[..., GROUP == i].sum(-1) for i in range(len(FEH_CENTRES))], -1)

### Load the data
The BOSS and APOGEE mass samples and their common parent through `lmcold.mass_samples.samples()` (built from the catalogues of
Zenodo bundle 1), and the APOGEE field of each covered candidate from `data/apogee_weights.parquet`.

In [ ]:
# lmcold.mass_samples.samples(): BOSS and APOGEE giants (calibrated [Fe/H], g, snr, weight w) and the source_id of the common parent
b, a, common = samples()
# data/apogee_weights.parquet: APOGEE field name ('LMC<n>') of each covered candidate, indexed by source_id
field = pd.read_parquet(DATA / "apogee_weights.parquet").query("covered").set_index("source_id").field

### Stars observed by both surveys
Calibrated BOSS minus APOGEE [Fe/H] of the stars with both spectra, and its median per reporting bin of APOGEE [Fe/H]
(panel a).

In [ ]:
# Overlap residuals per reporting bin of APOGEE [Fe/H] (panel a, and `bias` in the json).
o = overlap(b, a)   # stars in both samples (same Gaia source): feh_b (BOSS), feh_a (APOGEE) [dex], g, snr, ra, dec
d = (o.feh_b - o.feh_a).to_numpy()   # (N,) BOSS - APOGEE [Fe/H] [dex]
k = np.digitize(o.feh_a, REPORT_EDGES) - 1   # reporting bin of each star's APOGEE [Fe/H]
# Number of overlap stars and median residual per reporting bin (NaN for an empty bin).
bias = pd.DataFrame(dict(feh=FEH_CENTRES, n=[int(np.sum(k == i)) for i in range(len(FEH_CENTRES))],
                         median=[float(np.median(d[k == i])) if np.any(k == i) else np.nan for i in range(len(FEH_CENTRES))]))
print(bias.round(3).to_string(index=False))

### The common parent and the closure test
Both samples restricted to the parent common to the two surveys, with the APOGEE fields LMC1-17 flagged; the bootstrap draws
of nside-16 sky pixels; and the closure test: observed BOSS counts against the APOGEE counts passed through the response.

In [ ]:
# Restrict both samples to the common parent; wd: APOGEE field LMC1-17 (field names 'LMC<n>'), Washington + DDO51 targeting.
bc, ac = b[b.source_id.isin(common)].copy(), a[a.source_id.isin(common)].copy()
bc["wd"], ac["wd"] = bc.source_id.map(field).str[3:].astype(int) <= 17, ac.field.str[3:].astype(int) <= 17
# nside-16 nested HEALPix pixel of each star (the bootstrap resamples these pixels).
pix_b = hp.ang2pix(16, bc.ra.to_numpy(), bc.dec.to_numpy(), lonlat=True, nest=True)
pix_a = hp.ang2pix(16, ac.ra.to_numpy(), ac.dec.to_numpy(), lonlat=True, nest=True)
# Bootstrap: each row of draws gives how many times each nside-16 pixel (of either sample) is drawn.
upix = np.unique(np.concatenate([pix_b, pix_a]))   # sorted pixels occupied by either sample
rng = np.random.default_rng(20261027)   # fixed seed
# (NBOOT, len(upix)): multinomial draw of len(upix) pixels with equal probability, i.e. resampling with replacement.
draws = rng.multinomial(len(upix), np.ones(len(upix)) / len(upix), size=NBOOT)


def per_pixel(t, pix, edges, sel):
    """Weighted counts per pixel and [Fe/H] bin.

    Parameters
    ----------
    t : pandas.DataFrame
        Sample with columns feh [dex] and w (selection weight).
    pix : numpy.ndarray
        nside-16 nested pixel of each row of t.
    edges : array_like
        [Fe/H] bin edges; values are clipped into [-2.999, 0.499] first, so the end bins collect the tails.
    sel : numpy.ndarray of bool
        Rows to use.

    Returns
    -------
    numpy.ndarray
        Shape (len(upix), len(edges) - 1).
    """
    m = np.zeros((len(upix), len(edges) - 1))
    # [Fe/H] bin of each selected star, then its weight added to its (pixel, bin) cell.
    j = np.digitize(np.clip(t.feh.to_numpy()[sel], -2.999, 0.499), edges) - 1
    np.add.at(m, (np.searchsorted(upix, pix[sel]), j), t.w.to_numpy()[sel])
    return m


def closure(sel_b, sel_a, o_fit):
    """Observed and predicted BOSS counts in the reporting bins, with joint bootstrap draws (pixels and overlap stars).

    Parameters
    ----------
    sel_b, sel_a : numpy.ndarray of bool
        Rows of the common-parent BOSS (bc) and APOGEE (ac) samples to use.
    o_fit : pandas.DataFrame
        Overlap stars the response is measured on.

    Returns
    -------
    dict
        chi2 and dof over bins with >= 1 weighted star in APOGEE or BOSS; per reporting bin the observed, predicted and APOGEE
        counts and the 16th/84th percentiles of observed / predicted over the draws; below_-2.0 and below_-1.5 totals.
    """
    # Per-pixel weighted counts: BOSS in the fine measured bins, APOGEE in the true bins; APOGEE in the reporting bins (all sky).
    mb, ma = per_pixel(bc, pix_b, MEASURED_EDGES, sel_b), per_pixel(ac, pix_a, TRUE_EDGES, sel_a)
    direct = per_pixel(ac, pix_a, REPORT_EDGES, sel_a).sum(0)
    t, w = bc[sel_b], bc.w.to_numpy()[sel_b]   # BOSS target sample and its weights, for the G weighting of the response
    # Observed BOSS counts, and the prediction R @ (APOGEE counts per true bin), both in the reporting bins.
    obs, pred = coarse(mb.sum(0)), coarse(response(o_fit, t, w) @ ma.sum(0))
    dobs = coarse(draws @ mb)  # (NBOOT, reporting bins)
    # Each draw also resamples the overlap stars with replacement before measuring the response.
    dpred = np.array([coarse(response(o_fit.iloc[rng.integers(0, len(o_fit), len(o_fit))], t, w) @ (dr @ ma)) for dr in draws])
    use = (direct >= 1) | (obs >= 1)   # bins entering chi2
    # Bootstrap covariance of observed - predicted; chi2 = r^T C^-1 r over the used bins.
    cov = np.cov((dobs - dpred)[:, use].T)
    with np.errstate(divide="ignore", invalid="ignore"):
        rb = dobs / dpred   # (NBOOT, reporting bins) observed / predicted per draw
    res = dict(chi2=float((obs - pred)[use] @ np.linalg.solve(cov, (obs - pred)[use])), dof=int(use.sum()),
               observed=obs.tolist(), predicted=pred.tolist(), apogee=direct.tolist(),
               ratio_p16=np.nanpercentile(rb, 16, axis=0).tolist(), ratio_p84=np.nanpercentile(rb, 84, axis=0).tolist())
    for x, n in ((-2.0, 1), (-1.5, 3)):  # the first n reporting bins lie below x
        rat = dobs[:, :n].sum(1) / dpred[:, :n].sum(1)   # (NBOOT,) ratio of the totals below x
        res[f"below_{x}"] = dict(observed=float(obs[:n].sum()), predicted=float(pred[:n].sum()), apogee=float(direct[:n].sum()),
                                 ratio=float(obs[:n].sum() / pred[:n].sum()), ratio_p16_p84=np.percentile(rat, [16, 84]).tolist())
    return res


def half(t):
    """Sky half of each row of t: parity (0 or 1) of its nside-8 nested HEALPix pixel."""
    return hp.ang2pix(8, t.ra.to_numpy(), t.dec.to_numpy(), lonlat=True, nest=True) % 2

### Closure on all stars, by field type and across the sky
All stars, LMC1-17 and LMC18-35 separately, and the response measured on one half of the sky and tested on the other;
written to `results/boss_response.json`.

In [ ]:
every_b, every_a = np.ones(len(bc), bool), np.ones(len(ac), bool)   # all rows of the common-parent samples
# Summary for the json: overlap size, residuals per bin, bin edges, and closure on all stars and per APOGEE field type.
out = dict(overlap=len(o), bias=bias.to_dict("list"), true_edges=TRUE_EDGES, measured_edges=MEASURED_EDGES.tolist(),
           closure={"all": closure(every_b, every_a, o),
                    "LMC1-17 (W+D)": closure(bc.wd.to_numpy(), ac.wd.to_numpy(), o),
                    "LMC18-35": closure(~bc.wd.to_numpy(), ~ac.wd.to_numpy(), o)})
# Split-sky test: response from the overlap stars of half h, closure on the stars of the other half.
for h in (0, 1):
    out["closure"][f"response from sky half {h}, tested on {1 - h}"] = closure(half(bc) == 1 - h, half(ac) == 1 - h, o[half(o) == h])
(ROOT / "results/boss_response.json").write_text(json.dumps(out, indent=1))
# Print chi2 and the totals below [Fe/H] = -2 and -1.5 of each closure case.
for name, c in out["closure"].items():
    print(f"{name}: chi2 {c['chi2']:.1f}/{c['dof']}; " + "; ".join(
        f"{x}: BOSS {c[x]['observed']:.1f} predicted {c[x]['predicted']:.1f} APOGEE {c[x]['apogee']:.1f} ratio {c[x]['ratio']:.2f} "
        f"[{c[x]['ratio_p16_p84'][0]:.2f}, {c[x]['ratio_p16_p84'][1]:.2f}]" for x in ("below_-2.0", "below_-1.5")))

### The figure

In [ ]:
# Figure: (a) overlap residuals against APOGEE [Fe/H]; (b) observed / predicted BOSS counts by APOGEE field type, bins with
# >= 3 weighted APOGEE stars.
plt.style.use(ROOT / "src/lmcold/paper.mplstyle")
fig, ax = panel_grid(1, 2, width=5.0, left=0.6, right=0.1, hgap=0.6, bottom=0.45, top=0.6)
blue, orange, green, grey = "#2878b5", "#d75a20", "#33976b", "0.6"
# Panel (a): every overlap star, and the median residual of bins with at least 3 stars.
a0 = ax[0, 0]
a0.plot(o.feh_a, d, ".", ms=1, color=grey, rasterized=True)
ok = bias.n >= 3
a0.plot(bias.feh[ok], bias["median"][ok], "o-", ms=3, color=blue, lw=0.8, label="median per bin")
a0.axhline(0, color="0.3", lw=0.6, ls="--")
a0.set(xlim=(-2.8, 0), ylim=(-0.8, 0.8), xlabel=r"[Fe/H]$_{\rm APOGEE}$", ylabel=r"[Fe/H]$_{\rm BOSS}-$[Fe/H]$_{\rm APOGEE}$",
       title="(a) Stars with both spectra")
a0.legend(loc="lower left", bbox_to_anchor=(0, 1.12), borderaxespad=0)   # legend above the panel
# Panel (b): observed / predicted per reporting bin with its 16-84th percentile range, field types offset by +-0.03 dex.
a1 = ax[0, 1]
for name, lab, col, off in (("LMC1-17 (W+D)", "LMC1–17 (W+D)", orange, -0.03), ("LMC18-35", "LMC18–35", green, 0.03)):
    c = out["closure"][name]
    obs, ref = np.array(c["observed"]), np.array(c["predicted"])
    good = (np.array(c["apogee"]) >= 3) & (ref > 0)   # bins with >= 3 weighted APOGEE stars
    a1.vlines(FEH_CENTRES[good] + off, np.array(c["ratio_p16"])[good], np.array(c["ratio_p84"])[good], color=col, lw=0.8)
    a1.plot(FEH_CENTRES[good] + off, obs[good] / ref[good], "o", ms=3, color=col, label=lab)
a1.axhline(1, color="0.3", lw=0.6, ls="--")
a1.set(xlim=(-2.45, -0.1), ylim=(0, 3.2), xlabel="[Fe/H]", xticks=[-2.25, -1.5, -1, -0.5], xticklabels=["<−2", "−1.5", "−1", "−0.5"],
       ylabel="BOSS observed / expected", title="(b) By APOGEE field type")
a1.legend(loc="lower left", bbox_to_anchor=(0, 1.12), borderaxespad=0, ncol=2)
fig.savefig(ROOT / "figures/paper_boss_response.png")

In [ ]:
from IPython.display import Image
Image(filename='figures/paper_boss_response.png')

**Figure 4.1.** The BOSS metallicity response. (a) Calibrated BOSS minus APOGEE [Fe/H] (dex) against APOGEE [Fe/H]
for the 1 923 stars of the mass samples observed by both surveys (grey points), with the median per
0.25-dex bin of APOGEE [Fe/H] (blue); dashed: zero. (b) Corrected BOSS counts in the common parent over the APOGEE
counts passed through the response, against [Fe/H], separately for the APOGEE fields LMC1–17 (orange; targeted with a Washington +
DDO51 giant flag) and LMC18–35 (green); bars: 16th–84th percentiles of 500 joint bootstraps of sky pixels and of
the stars observed by both; dashed: a ratio of 1. `scripts/boss_response.py`

### The formed-mass computation

`scripts/build_giant_mass.py` carries out the steps above for the whole BOSS sample. It measures the [α/Fe] trend, sets up the
conversion regions with their extinction and disc offset, computes the foreground fraction of each giant, counts the isochrone stars
inside the selection for every region and model alternative, and fits the masses $M_k$ through the response. It also draws the
bootstrap resamples and evaluates the model alternatives described under Uncertainties, and it computes the masses of BOSS and APOGEE
in the field cores used in the checks below. Its tables in `results/` feed every mass figure of this page and the mass per
metallicity of Letter Fig. 1b.

<details><summary>About <code>scripts/build_giant_mass.py</code></summary>

```text
Stellar mass formed per [Fe/H] bin from the BOSS giants (paper Sect. 2.3; Letter Fig. 1b through build_present_mass.py),
and the BOSS / APOGEE comparison in shared field cores (appendix figure paper_mass_cores).

The weighted BOSS giant counts per [Fe/H] bin are turned into formed mass by dividing by eta, the number of selected giants
per Msun formed computed from PARSEC isochrones (lmcold.isochrones, lmcold.mass_conversion), region by region, after removing
the Milky Way foreground and correcting for the BOSS [Fe/H] response. The KB chapter "Stellar mass per metallicity" describes
each step and its equations.

Four parts of the method go beyond a weighted count divided by one conversion:
- Above [Fe/H] = -1 the regional BOSS counts are shared among the reporting bins as the APOGEE giants of the same ring x G x
  dereddened-colour cell (lmcold.mass_samples.metal_rich_split): BOSSNet over-estimates [Fe/H] of hot, low-S/N giants, which the
  stars with both spectra barely contain (scripts/paper_boss_hot_bias.py). BOSS still sets the total above -1. The field cores
  use the same split, so above -1 only their total is an independent test.
- BOSS metallicity response measured on the stars with both spectra (lmcold.mass_samples.boss_response, closure tested in
  scripts/boss_response.py): masses per true [Fe/H] bin from a non-negative Poisson-mean fit (lmcold.mass_response.recover_mass)
  of the BOSS counts in fine measured bins into 0.5-dex true bins, with an effective conversion per bin (count-weighted harmonic
  mean over regions and reporting bins); regional masses are the direct ones (measured bins) times the ratio of fitted to
  direct mass of their true bin.
- Conversion per region with the true reddening (lmcold.extinction) and the inclined-disc distance (lmcold.geometry.disc_dmu),
  besides the reddening the carton removed: the full BOSS sample in 16 regions (rings 0-3-6-9-12 deg x quadrants), each at
  the median values of its parent stars; APOGEE and BOSS field cores at the medians over the core.
- Milky Way foreground varying with radius (lmcold.foreground), per reporting bin, normalised to the transferred reference mean
  fraction over 2-10 deg; bracketed by zero and twice its amplitude.
Model alternatives: age weightings AMR (reference; lmcold.mass_conversion) and windows A/B/C x alpha (measured, 0, 0.15, 0.3) x foreground (0, 1, 2); and, for the reference, the
Skowron et al. E(V-I) systematic (+-0.025 mag) and the vdMK14 inclination error (34 +- 7 deg). Non-overlapping APOGEE cores
r<0.8 deg, contiguous >=90%-covered magnitude chunks, H_hi>=13.3. Full BOSS coverage fill (lmcold.selection.coverage_factor) is
an explicit ring (0-3-6-12 deg) x G extrapolation. Bootstrap: NBOOT nside-16 pixel draws of the parent, selection weights and coverage recomputed per draw, overlap stars of the
response resampled.

Inputs. sources.yaml entries: apogee_dr17_allstarlite (ALPHA_M for the alpha trend), apogee2_field and apogee2_object_<field>
(core centres; SFD_EBV of the core stars), skowron21_table1 (via lmcold.extinction.rc_map), mass_foreground_reference,
parsec_gaiaEDR3 (BOSS) and parsec_2mass_spitzer (APOGEE). Data files: boss_weights, boss_parent, apogee_parent,
apogee_weights (release_* entries) and, through lmcold.mass_samples.samples, spec_giants.parquet and
results/boss_calibration.json.
Units: masses in Msun formed (Kroupa IMF, 0.08-100 Msun); eta in giants per Msun formed; E(B-V) [mag, SFD scale]; dmu [mag];
[Fe/H] in dex on the APOGEE scale; reporting bins lmcold.mass_samples.FEH_EDGES (centres FEH_CENTRES, -2.25 ... +0.25).
Outputs (results/ unless stated):
  mass_alpha_trend.csv            median [Fe/H], [alpha/M] and N per reporting bin with >= 10 APOGEE giants;
  mass_fields_apogee_candidates.csv, mass_fields.csv, mass_fields_rejected.csv
                                  APOGEE field cores (h_limit, maglim, centre, ebv_carton, ebv, dmu, boss_coverage);
  mass_regions.csv                the 16 BOSS regions: n_parent, median true/carton E(B-V), dmu (and at inc -+ 7 deg), rho;
  data/giant_foreground.parquet   per BOSS giant: source_id, feh, rho, fill, f_mw (foreground fraction);
  giant_mass_variants.csv         mass per reporting bin for every model alternative;
  giant_mass_response.csv         reference: direct, fitted and fitted/direct per bin;
  mass_tail_conversion.csv        eta at -2.5, -2.25, -2.0 and whether [M/H] is clipped at the grid edge;
  giant_mass_fields.csv, mass_apogee_fill.csv
                                  per field core and survey: counts, raw counts, eta, mass; APOGEE fill per H chunk;
  mass_conversion_factors.csv     factors() output per survey x field/region x prior x alpha;
  mass_model_grid.csv             giants per Msun formed on the isochrone grid for every conversion;
  giant_mass_bootstrap.csv        NBOOT draws of the reference mass per bin;
  giant_mass.csv                  reference mass, cumulative mass, bootstrap 16-84%, model min/max, covered and direct mass;
  giant_mass_common.csv           BOSS and APOGEE masses in the common cores, ratio and its 16-84% over 500 paired draws;
  giant_mass_assignment.csv       change of the direct mass if faint giants below -2 take their XP [M/H] instead;
  giant_mass.json                 summary numbers and definitions.
These feed build_present_mass.py, figures/paper_mass.png, paper_mass_cores.png, paper_mass_summary.png, and (through
giant_foreground.parquet) paper_feh_profile.py, fig_giant_foreground.py and other figure scripts.

Run: .venv/bin/python scripts/build_giant_mass.py   (after boss_selection.py and apogee_selection.py)
Run time: 162 s on a MacBook (M-series), docs/run_times.txt.
```
</details>

**Data read by `scripts/build_giant_mass.py`** (from the run log)

| File | Where it comes from | What it is |
|---|---|---|
| `data/apogee_parent.parquet` | Zenodo bundle 1 | APOGEE-2 LMC1-35 RGB-polygon candidates (field x star) with Gaia DR3 (2MASS best neighbour), observed flags and allStarLite columns |
| `data/apogee_weights.parquet` | Zenodo bundle 1 | APOGEE selection weights per candidate star (selection groups, coverage, w = N_cand / N_quality x pass factor in H x XP [M/H]) |
| `data/boss_parent.parquet` | Zenodo bundle 1 | Gaia DR3 stars within 12 deg of the LMC passing the rough cuts of mwm_magcloud_rgb_boss, with SFD E(B-V) and `eligible` |
| `data/boss_weights.parquet` | Zenodo bundle 1 | BOSS selection weights per eligible parent star (selection groups, w = N_elig / N_good x pass factor in G x XP [M/H] x ring) |
| `data/raw/parsec/2mass_spitzer.parquet` | public archive | PARSEC v1.2S + COLIBRI S_LMC_08, Kroupa IMF, 2mass_spitzer; earlier download checked against a fresh public slice at [M/H]=-1 |
| `data/raw/parsec/gaiaEDR3.parquet` | public archive | PARSEC v1.2S + COLIBRI S_LMC_08, Kroupa IMF, gaiaEDR3; earlier download checked against a fresh public slice at [M/H]=-1 |
| `data/raw/skowron21/table1.dat.gz` | public archive | OGLE-IV red-clump E(V-I) map of the Magellanic Clouds, 81,684 boxes of 3.4-27.5 arcmin; Skowron et al. 2021, ApJS 252, 23 (2021ApJS..252...23S, doi:10.3847/1538-4365/abcb81) |
| `data/spec_giants.parquet` | Zenodo bundle 1 | APOGEE DR17 and SDSS-V DR20 BOSS spectra of LMC giant candidates before the Gaia match |
| `results/boss_calibration.json` | repository | calibration of the BOSS [Fe/H] and velocities to APOGEE on the stars observed by both; written by `python scripts/build_giants.py` |
| `results/mass_foreground_reference.csv` | repository | Fixed input from an earlier analysis (not rebuilt here): in-field Milky Way foreground fractions per [Fe/H] bin (feh_med, f_mw, f_mw_err); transferred reference, not a new weighted membership fit; the transfer sensitivity uses zero and twice the reference amplitude |

*Shown, not run here: the selection-function catalogues of bundle 1, APOGEE DR17 files and PARSEC grids; 162 s. The cells below are the script itself; run it from the repository with* `python scripts/build_giant_mass.py`*.*

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

#### Settings
Modules, the output folder, the bin and ring edges of the conversion, the model alternatives and the size of the bootstrap.

```python
import json
import healpy as hp
import numpy as np
import pandas as pd
from astropy.io import fits
from lmcold import magcloud, selection
from lmcold.constants import LMC
from lmcold.extinction import EVI_PER_EBV, rc_map, true_ebv
from lmcold.footprint import angular_distance, lmc_xy
from lmcold.foreground import radial_fraction
from lmcold.geometry import disc_dmu
from lmcold.groups import bin_index
from lmcold.isochrones import count_grid, smooth_age_spikes
from lmcold.mass_conversion import REFERENCE_PRIOR, factors
from lmcold.mass_response import recover_mass
from lmcold.mass_samples import (samples, boss_response, overlap, metal_rich_split, with_colour, FEH_EDGES, FEH_CENTRES,
                                 MEASURED_EDGES, TRUE_EDGES)
from lmcold.sources import DATA, ROOT, path, sha256

OUT=ROOT/'results'  # output folder of the tables
N=len(FEH_CENTRES)  # number of reporting [Fe/H] bins (10)
G_EDGES=[12,15.5,16.25,16.75,17.1,17.5]  # G bins [mag] of the BOSS coverage fill (as boss_selection.GEDGES)
H_EDGES=np.array([12.35,12.8,13.3,13.8,14.3,14.8,15.3])  # H chunks [mag] of the APOGEE cores, BrtRGB bright limit to 15.3
RINGS=[0,3,6,9,12]  # deg: ring edges of the 16 BOSS conversion regions (4 rings x 4 quadrants)
PRIORS,ALPHAS=[REFERENCE_PRIOR,'A','B','C'],['measured','0','0.15','0.3']  # age weightings; alpha models ('0.15': [alpha/Fe])
DEBV=.025/EVI_PER_EBV  # mag: Skowron et al. 2021 systematic (0.025 mag in E(V-I)) of the intrinsic RC colour, on the E(B-V) scale
DINC=7.  # deg: error of the disc inclination, van der Marel & Kallivayalil 2014
NBOOT=500  # bootstrap draws of the full BOSS mass
```

#### Helpers
`dmu` gives the distance-modulus offset of the inclined disc at a sky position, `hist` the weighted counts per reporting
[Fe/H] bin, and `fit` the response-corrected mass per bin from the counts and the conversions (lmcold.mass_response).

```python
def dmu(ra,dec,inc=LMC['inc']):
    """Distance-modulus offset [mag] of the LMC disc plane at (ra, dec) [deg] relative to the centre (lmcold.geometry.disc_dmu).

    Parameters
    ----------
    ra, dec : array_like
        Positions [deg].
    inc : float
        Disc inclination [deg]; the line of nodes is constants.LMC['theta'].

    Returns
    -------
    numpy.ndarray
        dmu = 5 log10(d / D0) [mag], positive on the far side of the disc.
    """
    # All angles to radians: position, inclination, line of nodes and the adopted centre (constants.LMC)
    return np.asarray(disc_dmu(np.radians(np.asarray(ra,float)),np.radians(np.asarray(dec,float)),np.radians(inc),np.radians(LMC['theta']),
                               np.radians(LMC['ra0']),np.radians(LMC['dec0'])))


def hist(s,w):
    """Weighted counts of sample s (column feh [dex]) in the reporting bins FEH_EDGES; w: weight per row."""
    return np.histogram(s.feh,FEH_EDGES,weights=w)[0]


def fit(regional,fine,eta,r):
    """Response-corrected mass per reporting bin, the ratio fitted/direct per reporting bin, and the direct mass, from counts
    per region (regions x reporting bins), the BOSS counts in the fine measured bins, conversions per region (regions x
    reporting bins) and the response r (0.5-dex true bins TRUE_EDGES). Within a true bin the mass is shared out as the direct
    masses of its reporting bins."""
    direct=(regional/eta).sum(0)  # mass [Msun] = counts / (giants per Msun), summed over regions
    # tb: true bin of each reporting bin; per true bin, count-weighted harmonic-mean eta = sum(counts) / sum(direct mass).
    tb=np.searchsorted(TRUE_EDGES,FEH_CENTRES)-1
    c,d=np.bincount(tb,regional.sum(0),len(TRUE_EDGES)-1),np.bincount(tb,direct,len(TRUE_EDGES)-1)
    # Poisson-mean fit of the fine counts through the response into true-bin masses m [Msun] (eta.mean() where a
    # true bin has no direct mass); ratio: fitted / direct mass of each reporting bin's true bin
    m=recover_mass(fine,r,np.where(d>0,c/np.where(d>0,d,1),eta.mean()))
    ratio=np.where(d>0,m/np.where(d>0,d,1),0.)[tb]
    return direct*ratio,ratio,direct
```

#### Load the data
The BOSS and APOGEE mass samples and the stars observed by both (lmcold.mass_samples), the BOSS selection weights and parent,
the APOGEE parent and weights (Zenodo bundle 1), and the red-clump reddening map. Further inputs are read where they are used:
APOGEE allStarLite and the field list below, the foreground reference and the PARSEC grids further down.

```python
# BOSS (b) and APOGEE (a) mass samples: good giant spectra, positive weight, 150 < v_los < 420 km/s (lmcold.mass_samples).
b,a,_=samples()
o=overlap(b,a)  # stars with both spectra: feh_b (calibrated BOSS), feh_a (APOGEE) [dex], g [mag], snr
bw=pd.read_parquet(DATA/'boss_weights.parquet')  # BOSS eligible parent: selection group, good/observed flags, g, mh_xp, ra, dec
bp=pd.read_parquet(DATA/'boss_parent.parquet').query('eligible')  # SFD E(B-V) of the eligible parent stars
ap=pd.read_parquet(DATA/'apogee_parent.parquet')  # APOGEE candidates: field, h [mag], rfield [deg from field centre], maglim
# APOGEE weights joined with field, distance from the field centre and maglim; the same columns added to the sample
aw=pd.read_parquet(DATA/'apogee_weights.parquet').merge(ap[['field','apogee_id','rfield','maglim']],on=['field','apogee_id'],validate='one_to_one')
a=a.merge(ap[['field','apogee_id','rfield']],on=['field','apogee_id'],validate='one_to_one')
rc=rc_map()  # Skowron et al. 2021 red-clump E(V-I) boxes
```

#### The [alpha/Fe] trend
Median ASPCAP [alpha/M] of the APOGEE mass sample per reporting [Fe/H] bin (bins with at least 10 giants), used to convert
[Fe/H] to the model [M/H] (results/mass_alpha_trend.csv).

```python
# Abundance trend: exactly selected APOGEE stars, best nonduplicate spectrum.
with fits.open(path('apogee_dr17_allstarlite'),memmap=True) as h:
    d=h[1].data
    # Rows of the mass sample observed with the du Pont telescope (lco25m), EXTRATARG bit 4 (duplicate) not set
    ids=np.char.strip(d['APOGEE_ID'].astype(str))
    mask=pd.Series(ids).isin(a.apogee_id).to_numpy()&(np.char.strip(d['TELESCOPE'].astype(str))=='lco25m')&((d['EXTRATARG'] & (1<<4))==0)
    al=pd.DataFrame(dict(apogee_id=ids[mask],alpha=d['ALPHA_M'][mask].astype(float),snr=d['SNR'][mask].astype(float)))
# One row per star: the highest-S/N spectrum; joined with the sample's [Fe/H]
al=al.sort_values('snr',ascending=False,kind='stable').drop_duplicates('apogee_id')
al=a[['apogee_id','feh']].merge(al,on='apogee_id',validate='one_to_one')
al=al[np.isfinite(al.alpha)&(abs(al.alpha)<2)]  # drop missing and out-of-range [alpha/M]
# Median [Fe/H], [alpha/M] and count per reporting bin; bins with >= 10 giants kept
trend=al.groupby(pd.cut(al.feh,FEH_EDGES,right=False),observed=True).agg(feh=('feh','median'),alpha=('alpha','median'),n=('alpha','size')).query('n>=10').reset_index(drop=True)
trend.to_csv(OUT/'mass_alpha_trend.csv',index=False)
print('Alpha trend:',trend.to_string(index=False),flush=True)
```

#### APOGEE field cores
Cores of radius 0.8 deg about the APOGEE field centres where at least 90 per cent of the candidates have spectra down to
H = 13.3 or fainter; for each, the median carton and true E(B-V) and disc offset of its targeting-file stars.

```python
# Choose contiguous well-covered APOGEE cores.
# A core is r < 0.8 deg of an APOGEE field; walk up in H chunks from 12.35 while >= 90% of the candidates are covered (and below
# the field's maglim); keep fields whose covered range reaches H = 13.3 (h_limit = the last chunk's upper edge).
fields=[]
for field,q in aw[aw.rfield<.8].groupby('field'):
    # hi: upper H edge [mag] of the covered range so far, starting at the bright limit
    hi=12.35
    for low,high in zip(H_EDGES[:-1],H_EDGES[1:]):
        high=min(high,float(q.maglim.iloc[0]))
        part=q[(q.h>=low)&(q.h<high)]
        if high<=low or not len(part) or part.covered.mean()<.9:
            break
        hi=high
    # Keep the field if the covered range reaches H = 13.3
    if hi>=13.3:
        fields.append(dict(field=field,h_limit=hi,maglim=float(q.maglim.iloc[0])))
f=pd.DataFrame(fields)
# Field centres (ra, dec [deg]) from the apogee2_field table, one row per field name
with fits.open(path('apogee2_field')) as h:
    d=h[1].data
    centres=pd.DataFrame(dict(field=np.char.strip(d['FIELD_NAME'].astype(str)),ra=d['RA'].astype(float),dec=d['DEC'].astype(float))).drop_duplicates('field')
f=f.merge(centres,on='field',validate='one_to_one')
for i,r in f.iterrows():
    assert np.all(angular_distance(f.ra.drop(i),f.dec.drop(i),r.ra,r.dec)>1.6)  # 0.8-deg cores do not overlap
    # Median carton E(B-V), true E(B-V) and disc dmu over all targeting-file stars in the core.
    with fits.open(path('apogee2_object_'+r.field.lower())) as h:
        d=h[1].data
        # Targeting-file stars within 0.8 deg of the field centre: positions [deg] and SFD E(B-V) [mag]
        core=angular_distance(d['RA'],d['DEC'],r.ra,r.dec)<.8
        ra,dec,sfd=d['RA'][core].astype(float),d['DEC'][core].astype(float),d['SFD_EBV'][core].astype(float)
        f.loc[i,'ebv_carton']=np.nanmedian(magcloud.carton_ebv(ra,dec,sfd))
        f.loc[i,'ebv']=np.nanmedian(true_ebv(ra,dec,sfd,rc))
        f.loc[i,'dmu']=np.median(dmu(ra,dec))
```

#### BOSS coverage of the cores
Cores are dropped until in every remaining one at least 90 per cent of the BOSS eligible stars lie in selection groups with
enough good spectra inside the cores (results/mass_fields.csv, mass_fields_rejected.csv).

```python
# Require BOSS coverage too. Recompute after pruning because groups can cross
# field boundaries. This avoids estimating a field from a handful of stars.
f.to_csv(OUT/'mass_fields_apogee_candidates.csv',index=False)
s=pd.factorize(bw.group)[0]  # integer selection group of each eligible BOSS star
good=bw.good.to_numpy();obs=bw.observed.to_numpy()  # good giant spectrum; observed by BOSS
pg=selection.boss_pass_groups(bw.g,bw.mh_xp,bw.ra,bw.dec)  # groups for the pass factor (lmcold.selection)
# Drop cores where < 90% of the BOSS eligible stars lie in selection groups with >= NMIN_GOOD good spectra inside the cores.
rejected=[]
while True:
    # region: index of the core holding each eligible BOSS star (-1 outside every core)
    region=np.full(len(bw),-1)
    for i,r in f.iterrows():
        region[angular_distance(bw.ra,bw.dec,r.ra,r.dec)<.8]=i
    inside=region>=0
    # Good spectra inside the cores in each star's group; cov: group has >= NMIN_GOOD; coverage: covered fraction per core
    ng=np.bincount(s,inside*good)[s]
    cov=ng>=selection.NMIN_GOOD
    coverage=np.array([cov[region==i].mean() for i in range(len(f))])
    f['boss_coverage']=coverage
    # Stop when every core is >= 90 per cent covered; else record and drop the failing cores and recompute
    if np.all(coverage>=.9):break
    rejected.extend(f.loc[coverage<.9,['field','boss_coverage']].to_dict('records'))
    f=f[coverage>=.9].reset_index(drop=True)
    assert len(f)>0
f.to_csv(OUT/'mass_fields.csv',index=False)
pd.DataFrame(rejected).to_csv(OUT/'mass_fields_rejected.csv',index=False)
print('Common fields:',f.to_string(index=False),flush=True)
```

#### BOSS weights in the cores
Selection weights recomputed from the core stars only, a coverage fill per core and G bin, and the share of each BOSS giant
above [Fe/H] = -1 among the reporting bins, taken from the APOGEE giants of the same cell.

```python
# Local BOSS weights: group counts and pass factors from the core stars only (multiplicity = inside); fill per core x G bin.
wlocal=selection.group_weights(s,good,inside)*selection.pass_factor(s,obs,good,pg,inside)
gbin=bin_index(bw.g,G_EDGES)  # G bin of each eligible star
# Per eligible star in a core: core index, local weight and coverage fill per core x G cell; joined to the BOSS mass sample
local=bw[['source_id','g']].assign(region=region,local_w=wlocal,fill=selection.coverage_factor([region*10+gbin],cov,1,inside.astype(float)))[inside]
bc=b.drop(columns='g').merge(local,on='source_id',validate='one_to_one')
bc=bc[bc.local_w>0]
SPLIT_COLS=[f'split{k}' for k in range(5)]  # shares of the reporting bins above -1 (lmcold.mass_samples.metal_rich_split)
# Shares of each core giant among the reporting bins above -1, from the APOGEE giants of its cell
bc[SPLIT_COLS]=metal_rich_split(with_colour(bc[['source_id','ra','dec','feh']]),with_colour(a[['source_id','ra','dec','feh','w']]))


def split_hist(t,w):
    """Weighted counts per reporting bin, the giants above -1 shared as their split columns."""
    w=np.asarray(w,float)
    h=np.histogram(t.feh,FEH_EDGES,weights=w*(t.feh<=-1))[0]  # giants at or below -1 by their own [Fe/H]
    # Giants above -1 added to the last len(SPLIT_COLS) reporting bins as weight x share
    h[N-len(SPLIT_COLS):]+=(w[:,None]*t[SPLIT_COLS].to_numpy()).sum(0)
    return h
```

#### Conversion regions of the full BOSS sample
Coverage fill per ring and G bin, and the 16 conversion regions (four rings by four quadrants) with the median true and carton
E(B-V) and disc offset of their parent stars (results/mass_regions.csv).

```python
# Full BOSS: coverage fill in rings x G, conversion per ring x quadrant region.
rho=angular_distance(bw.ra,bw.dec,LMC['ra0'],LMC['dec0'])
assert rho.max()<12.01
rings=np.digitize(rho,[3,6])  # 0, 1, 2 for 0-3, 3-6, 6-12 deg (coverage fill)
# Covered selection groups (>= NMIN_GOOD good spectra); fill = parent / covered parent per ring x G cell.
fullcov=bw.group.map(bw.groupby('group').good.sum()).to_numpy()>=selection.NMIN_GOOD
ones=np.ones(len(bw))
# Coverage fill per eligible star (cells ring x G, at least 1 covered star per cell)
bw['fill']=selection.coverage_factor([rings*10+gbin],fullcov,1,ones)
x,y=lmc_xy(bw.ra.to_numpy(),bw.dec.to_numpy())  # deg, tangent-plane coordinates about the LMC centre (quadrants)
# Per eligible star: SFD E(B-V) (boss_parent), carton and true E(B-V), disc dmu at inc and inc -+ DINC; region 0-15.
geo=bw[['source_id','ra','dec']].merge(bp[['source_id','ebv']],on='source_id',validate='one_to_one')
geo=geo.assign(rho=rho,region=4*np.clip(np.digitize(rho,RINGS)-1,0,3)+(x>=0)+2*(y>=0),ebv_carton=magcloud.carton_ebv(geo.ra,geo.dec,geo.ebv),
               ebv_true=true_ebv(geo.ra.to_numpy(),geo.dec.to_numpy(),geo.ebv.to_numpy(),rc),dmu=dmu(geo.ra,geo.dec),
               dmu_lo=dmu(geo.ra,geo.dec,LMC['inc']-DINC),dmu_hi=dmu(geo.ra,geo.dec,LMC['inc']+DINC))
# Per region: parent count and median true and carton E(B-V) [mag], dmu (and at inc -+ DINC) [mag], rho [deg]
regions=geo.groupby('region').agg(n_parent=('ra','size'),ebv=('ebv_true','median'),ebv_carton=('ebv_carton','median'),
                                  dmu=('dmu','median'),dmu_lo=('dmu_lo','median'),dmu_hi=('dmu_hi','median'),rho=('rho','median'))
regions.to_csv(OUT/'mass_regions.csv')
print('Regions:',regions.round(3).to_string(),flush=True)
# Attach fill, rho and region to each BOSS giant of the mass sample
b=b.merge(bw[['source_id','fill']],on='source_id',validate='one_to_one').merge(geo[['source_id','rho','region']],on='source_id',validate='one_to_one')
```

#### Milky Way foreground
The foreground fraction f_mw of each BOSS giant: the reference mean fraction at its [Fe/H], spread in radius as a foreground of
uniform surface density per reporting bin (lmcold.foreground); written per giant to data/giant_foreground.parquet.

```python
# Foreground: reference mean fraction f_mw at [Fe/H] (interpolated in feh_med), spread in radius by lmcold.foreground.radial_fraction.
fg=pd.read_csv(path('mass_foreground_reference'))  # mean foreground fraction f_mw over 2-10 deg against feh_med
j=np.digitize(b.feh,FEH_EDGES)-1  # reporting bin of each BOSS giant
# Foreground fraction per giant, filled per reporting bin below
b['f_mw']=0.
for k in range(N):  # per reporting bin, so the surface-density profile is that of the same metallicities
    m=j==k
    if m.any():
        # Radial foreground fraction from the giants' radii [deg], weights w x fill and the reference mean fraction at their [Fe/H]
        b.loc[m,'f_mw']=radial_fraction(b.rho[m].to_numpy(),(b.w*b['fill'])[m].to_numpy(),np.interp(b.feh[m],fg.feh_med,fg.f_mw))
# Per-giant foreground table for the figure scripts
b[['source_id','feh','rho','fill','f_mw']].to_parquet(DATA/'giant_foreground.parquet',index=False)
```

#### Giants per solar mass formed
The PARSEC grids are read and the selected giants per Msun formed are counted on them: once at the medians of the whole
parent, and per region for the reference and the E(V-I) and inclination variants. `factors` averages them over age and
converts [Fe/H] to [M/H] for each age weighting and [alpha/Fe] model, giving eta per region and reporting bin.

```python
# PARSEC grids per survey (Gaia EDR3 photometry for BOSS, 2MASS for APOGEE); models: every computed grid
grids={name:pd.read_parquet(path(key)) for name,key in [('BOSS','parsec_gaiaEDR3'),('APOGEE','parsec_2mass_spitzer')]}
models=[]
def model(survey,field,ebv,ebv_carton,dmu,h_limit=13.3,maglim=14.5):
    """Giants per Msun formed on the isochrone grid for one survey and line of sight; also appended to `models`.

    Parameters
    ----------
    survey : {'BOSS', 'APOGEE'}
        Selects the PARSEC grid and the selection box (lmcold.isochrones.count_grid).
    field : str
        Label stored with the grid ('full', 'R<region><variant>' or an APOGEE field name).
    ebv, ebv_carton : float
        True and carton-removed E(B-V) [mag, SFD scale].
    dmu : float
        Disc distance-modulus offset [mag].
    h_limit, maglim : float
        APOGEE H limit of the core and the field's FntRGB limit [mag] (used for APOGEE).

    Returns
    -------
    pandas.DataFrame
        Spike-smoothed grid (lmcold.isochrones.smooth_age_spikes): mh, age, n_box per grid point.
    """
    print('Integrating',survey,field,f'E(B-V) {ebv:.4f} carton {ebv_carton:.4f} dmu {dmu:+.3f}',flush=True)
    # Selected giants per Msun on the grid for this line of sight, with the age spikes smoothed; stored in models
    g=smooth_age_spikes(count_grid(grids[survey],survey,ebv,ebv_carton,dmu,h_limit,maglim))
    models.append(g.assign(survey=survey,field=field,ebv=ebv,ebv_carton=ebv_carton,dmu=dmu,h_limit=h_limit,maglim=maglim))
    return g

# 'full': one conversion at the medians of the whole parent, for the age/metallicity bookkeeping of build_present_mass and
# the tail check; the masses use the regional conversions.
fullgrid=model('BOSS','full',float(geo.ebv_true.median()),float(geo.ebv_carton.median()),float(geo.dmu.median()))
# Regional grids: reference ('') and the systematic variants E(B-V) +-DEBV and inclination -+DINC.
regional={name:[model('BOSS',f'R{r}{name}',e.ebv+de,e.ebv_carton,e[dm]) for r,e in regions.iterrows()]
          for name,de,dm in [('',0,'dmu'),(' E+',DEBV,'dmu'),(' E-',-DEBV,'dmu'),(' i-',0,'dmu_lo'),(' i+',0,'dmu_hi')]}
# factors() of every grid for every age weighting and alpha model (saved to mass_conversion_factors.csv)
conversion=[]
for prior in PRIORS:
    for alpha in ALPHAS:
        conversion.append(factors(fullgrid,trend,prior,alpha).assign(survey='BOSS',field='full',prior=prior,alpha_model=alpha))
        for r,g in zip(regions.index,regional['']):
            conversion.append(factors(g,trend,prior,alpha).assign(survey='BOSS',field=f'R{r}',prior=prior,alpha_model=alpha))
# eta[(prior, alpha, variant)]: (regions x reporting bins); variants other than '' only for the reference prior and alpha.
eta={(prior,alpha,name):np.array([factors(g,trend,prior,alpha).eta.to_numpy() for g in gs])
     for name,gs in regional.items() for prior in PRIORS for alpha in ALPHAS if name=='' or (prior,alpha)==(REFERENCE_PRIOR,'measured')}
```

#### The BOSS response and the metal-rich split
The response matrix measured on the stars observed by both surveys, weighted to the BOSS sample, and the share of each BOSS
giant above [Fe/H] = -1 among the reporting bins; `counts` sums the weighted giants per region and reporting bin.

```python
rid=np.searchsorted(regions.index.to_numpy(),b.region.to_numpy())  # row of each BOSS giant's region in `regions`
jf=np.digitize(b.feh,MEASURED_EDGES)-1  # fine measured bin of each BOSS giant
# BOSS response matrix from the overlap stars, weighted to the BOSS sample (weights w x fill)
response=boss_response(o,b.feh,b.g,b.snr,(b.w*b['fill']).to_numpy())
# Above [Fe/H] = -1 each BOSS giant is shared among the reporting bins as the APOGEE giants of its ring x G x colour cell
# (lmcold.mass_samples.metal_rich_split): BOSSNet over-estimates [Fe/H] of hot, low-S/N giants (scripts/paper_boss_hot_bias.py).
rich=(b.feh>-1).to_numpy()  # BOSS giants above [Fe/H] = -1, shared out by `split`
split=metal_rich_split(with_colour(b[['source_id','ra','dec','feh']]),with_colour(a[['source_id','ra','dec','feh','w']]))
assert len(split)==len(b)


def counts(weights):
    """BOSS counts per region x reporting bin (giants above -1 shared by `split`) and per fine measured bin.

    Parameters
    ----------
    weights : numpy.ndarray
        Weight per BOSS giant (selection weight x foreground and fill terms).

    Returns
    -------
    reg : numpy.ndarray
        (len(regions), N).
    fine : numpy.ndarray
        (len(MEASURED_EDGES) - 1,).
    """
    # Giants at or below -1 by their own bin; those above -1 shared over the last split.shape[1] bins; fine: all giants
    reg=np.zeros((len(regions),N))
    np.add.at(reg,(rid[~rich],j[~rich]),weights[~rich])
    np.add.at(reg[:,N-split.shape[1]:],rid[rich],weights[rich,None]*split[rich])
    return reg,np.bincount(jf,weights,len(MEASURED_EDGES)-1)
```

#### Mass for every model alternative
The fit for every combination of conversion, age weighting, [alpha/Fe] model, foreground amplitude (0, 1, 2) and coverage fill
(results/giant_mass_variants.csv), and the reference with its fitted-to-direct ratios (giant_mass_response.csv).

```python
# Every model alternative; the weight per giant is w x (1 - foreground x f_mw) x fill (if filled).
variants=[];direct_rows=[]
for (prior,alpha,name),e in eta.items():
    for foreground in ([0,1,2] if name=='' else [1]):
        for filled in ([False,True] if name=='' else [True]):
            # Counts with this foreground amplitude and fill, response-corrected mass per bin [Msun]; one table row per bin
            reg,fine=counts((b.w*(1-foreground*b.f_mw)*(b['fill'] if filled else 1)).to_numpy())
            m,ratio,direct=fit(reg,fine,e,response)
            variants.append(pd.DataFrame(dict(conversion=name.strip() or 'reference',prior=prior,alpha_model=alpha,foreground=foreground,
                                              filled=filled,feh=FEH_CENTRES,mass=m,cumulative=np.cumsum(m),direct_mass=direct)))
# All alternatives to giant_mass_variants.csv; ref: rows of the reference model (reference conversion, prior,
# measured alpha, foreground 1)
v=pd.concat(variants,ignore_index=True)
v.to_csv(OUT/'giant_mass_variants.csv',index=False)
ref=(v.conversion=='reference')&(v.prior==REFERENCE_PRIOR)&(v.alpha_model=='measured')&(v.foreground==1)
# Reference mass recomputed directly and checked against the filled reference row; then giant_mass_response.csv
reg_ref,fine_ref=counts((b.w*(1-b.f_mw)*b['fill']).to_numpy())
mass_ref,ratio_ref,direct_ref=fit(reg_ref,fine_ref,eta[REFERENCE_PRIOR,'measured',''],response)
np.testing.assert_allclose(mass_ref,v[ref&v.filled].mass.to_numpy())
pd.DataFrame(dict(feh=FEH_CENTRES,fitted_over_direct=ratio_ref,direct=direct_ref,fitted=mass_ref)).to_csv(OUT/'giant_mass_response.csv',index=False)
```

#### Conversion of the metal-poor tail
eta at [Fe/H] = -2.5, -2.25 and -2 against that of the first reporting bin, and whether [M/H] falls off the grid
(results/mass_tail_conversion.csv).

```python
# Conversion of the metal-poor tail: eta at single [Fe/H] values, relative to the reference first bin.
# reff: eta per reporting bin of the 'full' grid with the default prior and alpha
tail=[]
reff=factors(fullgrid,trend).eta.to_numpy()
for centre in [-2.5,-2.25,-2.]:
    # eta at this single [Fe/H], the grid [M/H] it maps to and whether it is clipped at the grid edge
    fac=factors(fullgrid,trend,centres=[centre])
    tail.append(dict(feh=centre,eta=fac.eta.iloc[0],mh_grid=fac.mh_grid.iloc[0],clipped=bool(fac.clipped_mh.iloc[0]),mass_ratio_to_reference=reff[0]/fac.eta.iloc[0]))
pd.DataFrame(tail).to_csv(OUT/'mass_tail_conversion.csv',index=False)
```

#### BOSS and APOGEE in the field cores
Each survey with its own selection box and the extinction and disc offset of each core; APOGEE counts filled per H chunk,
BOSS counts response-corrected with the response fitted to the summed core counts (results/giant_mass_fields.csv).

```python
# Field-specific conversions and counts, with local BOSS weights; the BOSS response fitted to the summed core counts.
rows=[];afills=[]
# Summed BOSS counts of all cores in the fine measured bins (local weights x fill), and the core response
core_fine=np.zeros(len(MEASURED_EDGES)-1)
for i,r in f.iterrows():
    bs=bc[bc.region==i]
    core_fine+=np.histogram(bs.feh,MEASURED_EDGES,weights=bs.local_w*bs['fill'])[0]
core_response=boss_response(o,bc.feh,bc.g,bc.snr,(bc.local_w*bc['fill']).to_numpy())
core={}
for i,r in f.iterrows():
    # Giants per Msun for APOGEE (no carton reddening removal: ebv twice) and for BOSS at this core
    ag=model('APOGEE',r.field,r.ebv,r.ebv,r.dmu,r.h_limit,r.maglim)
    bg=model('BOSS',r.field,r.ebv,r.ebv_carton,r.dmu)
    # APOGEE mass-sample giants of the core within its covered H range; raw: unweighted counts per reporting bin
    ac=a[(a.field==r.field)&(a.rfield<.8)&(a.h>=12.35)&(a.h<r.h_limit)].copy()
    acount=np.zeros(N);raw=hist(ac,np.ones(len(ac)))
    for low,high in zip(H_EDGES[:-1],H_EDGES[1:]):
        high=min(high,r.h_limit)
        if high<=low:break
        # Parent rows (astro_ok) and covered candidates of the core in this H chunk
        p=ap[(ap.field==r.field)&(ap.rfield<.8)&ap.astro_ok&(ap.h>=low)&(ap.h<high)]
        q=aw[(aw.field==r.field)&(aw.rfield<.8)&aw.covered&(aw.h>=low)&(aw.h<high)]
        fill=len(p)/len(q)  # astro_ok parent rows (bright-neighbour stars included) / covered candidates in the chunk
        afills.append(dict(field=r.field,h_lo=low,h_hi=high,parent=len(p),covered=len(q),fill=fill))
        # Giants of the chunk, weighted by w x fill
        part=ac[(ac.h>=low)&(ac.h<high)]
        acount+=hist(part,part.w*fill)
    # Per core: (grid, weighted counts, raw counts) of each survey; BOSS counts with the metal-rich split
    bs=bc[bc.region==i]
    core[r.field]=dict(APOGEE=(ag,acount,raw),BOSS=(bg,split_hist(bs,bs.local_w*bs['fill']),hist(bs,np.ones(len(bs)))))
for prior in PRIORS:
    for alpha in ALPHAS:
        # eta per core and survey; BOSS fitted/direct ratios from the summed core counts and the core response
        fac={(fld,survey):factors(c[survey][0],trend,prior,alpha) for fld,c in core.items() for survey in ('APOGEE','BOSS')}
        e=np.array([fac[fld,'BOSS'].eta.to_numpy() for fld in core])
        bcounts=np.array([core[fld]['BOSS'][1] for fld in core])
        _,bratio,_=fit(bcounts,core_fine,e,core_response)
        for fld,c in core.items():
            for survey in ('APOGEE','BOSS'):
                g,count,nraw=c[survey]
                conversion.append(fac[fld,survey].assign(survey=survey,field=fld,prior=prior,alpha_model=alpha))
                direct=count/fac[fld,survey].eta.to_numpy()
                mass=np.nan_to_num(direct*bratio) if survey=='BOSS' else direct  # APOGEE: no response correction
                rows.append(pd.DataFrame(dict(field=fld,survey=survey,prior=prior,alpha_model=alpha,feh=FEH_CENTRES,count=count,raw=nraw,
                                              eta=fac[fld,survey].eta.to_numpy(),mass=mass,direct_mass=direct)))
# Write the core masses, the APOGEE fills, the conversion factors and every isochrone grid
fieldmass=pd.concat(rows,ignore_index=True)
fieldmass.to_csv(OUT/'giant_mass_fields.csv',index=False)
pd.DataFrame(afills).to_csv(OUT/'mass_apogee_fill.csv',index=False)
pd.concat(conversion,ignore_index=True).to_csv(OUT/'mass_conversion_factors.csv',index=False)
pd.concat(models,ignore_index=True).to_csv(OUT/'mass_model_grid.csv',index=False)
```

#### Bootstrap and the reference mass table
NBOOT draws of nside-16 sky pixels of the parent, with the selection weights and coverage recomputed and the stars observed by
both resampled; percentiles and model ranges go with the reference into results/giant_mass.csv.

```python
# Bootstrap: nside-16 pixels of the parent, with the selection weights and coverage factors recomputed for each draw, and the
# overlap stars of the response; conversions fixed.
rng=np.random.default_rng(20261026)  # fixed seed
# nside-16 nested pixel of each eligible star; u: occupied pixels, ii: index of each star's pixel in u
pix=hp.ang2pix(16,bw.ra.to_numpy(),bw.dec.to_numpy(),lonlat=True,nest=True)
u,ii=np.unique(pix,return_inverse=True)
pos=pd.Index(bw.source_id).get_indexer(b.source_id)  # row of each BOSS giant in the parent
boot=[]
# Each draw: pixel multiplicities from a multinomial (resampling pixels with replacement); m: per-star multiplicity
for draw in rng.multinomial(len(u),np.full(len(u),1/len(u)),size=NBOOT):
    m=draw[ii].astype(float)
    # Selection weights and coverage fill recomputed with the multiplicities; wstar: weight x fill x m of each BOSS giant
    wb=selection.group_weights(s,good,m)*selection.pass_factor(s,obs,good,pg,m)
    fb=selection.coverage_factor([rings*10+gbin],np.bincount(s,m*good)[s]>=selection.NMIN_GOOD,1,m)
    wstar=(wb*fb*m)[pos]
    # Overlap stars resampled with replacement; mass per bin with the reference conversion and the redrawn response
    ob=o.iloc[rng.integers(0,len(o),len(o))]
    reg,fine=counts(wstar*(1-b.f_mw.to_numpy()))
    boot.append(fit(reg,fine,eta[REFERENCE_PRIOR,'measured',''],boss_response(ob,b.feh,b.g,b.snr,wstar))[0])
boot=np.array(boot)
pd.DataFrame(boot,columns=FEH_CENTRES).to_csv(OUT/'giant_mass_bootstrap.csv',index=False)
# Reference mass per bin [Msun] and cumulative mass, with bootstrap 16th-84th percentiles
summary=pd.DataFrame(dict(feh=FEH_CENTRES,feh_lo=FEH_EDGES[:-1],feh_hi=FEH_EDGES[1:],mass=mass_ref,cumulative=mass_ref.cumsum()))
summary[['mass_p16','mass_p84']]=np.percentile(boot,[16,84],axis=0).T
summary[['cumulative_p16','cumulative_p84']]=np.percentile(boot.cumsum(axis=1),[16,84],axis=0).T
# Model envelope: min and max over all filled alternatives; covered_mass: the reference without fill
for col in ['mass','cumulative']:
    summary[col+'_model_min']=v[v.filled].groupby('feh')[col].min().to_numpy()
    summary[col+'_model_max']=v[v.filled].groupby('feh')[col].max().to_numpy()
summary['covered_mass']=v[ref&~v.filled].mass.to_numpy()
summary['direct_mass']=direct_ref
summary.to_csv(OUT/'giant_mass.csv',index=False)
```

#### Survey ratio in the cores
BOSS over APOGEE mass per bin summed over the cores, with 16th-84th percentiles from 500 resamplings of the cores
(results/giant_mass_common.csv).

```python
# Paired resampling of the same field cores for the survey ratio.
# Reference model; mb, ma: BOSS and APOGEE mass [Msun] per (core, reporting bin)
q=fieldmass[(fieldmass.prior==REFERENCE_PRIOR)&(fieldmass.alpha_model=='measured')]
mb=q[q.survey=='BOSS'].pivot(index='field',columns='feh',values='mass').to_numpy()
ma=q[q.survey=='APOGEE'].pivot(index='field',columns='feh',values='mass').to_numpy()
# 500 resamplings of the cores (multiplicities); ratio of the summed masses per bin for each, and for the data
draw=rng.multinomial(len(f),np.full(len(f),1/len(f)),size=500)
with np.errstate(divide='ignore',invalid='ignore'):
    ratios=(draw@mb)/(draw@ma)
    ratio=mb.sum(axis=0)/ma.sum(axis=0)
# Bins with no APOGEE mass give NaN ratios
ratio[~np.isfinite(ratio)]=np.nan
ratios[~np.isfinite(ratios)]=np.nan
# Per bin: summed masses, ratio, BOSS direct mass and raw counts of both surveys
common=pd.DataFrame(dict(feh=FEH_CENTRES,boss_mass=mb.sum(axis=0),apogee_mass=ma.sum(axis=0),ratio=ratio,
                         boss_direct_mass=q[q.survey=='BOSS'].groupby('feh').direct_mass.sum().to_numpy()))
for survey in ['BOSS','APOGEE']:
    common[survey.lower()+'_raw']=q[q.survey==survey].groupby('feh').raw.sum().to_numpy()
# 16th-84th percentiles of the ratio where any draw is finite
common[['ratio_p16','ratio_p84']]=np.nan
has=np.any(np.isfinite(ratios),axis=0)
common.loc[has,['ratio_p16','ratio_p84']]=np.nanpercentile(ratios[:,has],[16,84],axis=0).T
common.to_csv(OUT/'giant_mass_common.csv',index=False)
```

#### Faint giants below [Fe/H] = -2
Change of the direct mass if the BOSS giants fainter than G = 17.1 below [Fe/H] = -2 take their Gaia XP [M/H] instead
(results/giant_mass_assignment.csv).

```python
# The identified faint-tail assignment comparison, on the direct masses (measured bins).
# Weight per star divided by eta of its region and (new) bin gives mass; foreground interpolated at the assigned [Fe/H].
sens=[]
e=eta[REFERENCE_PRIOR,'measured','']
# Two cases: giants in -2.25 <= [Fe/H] < -2, and all giants below -2 (G > 17.1, with XP [M/H])
for label,low in [('disputed_bin',-2.25),('all_faint_below_m2',-np.inf)]:
    changed=((b.g>17.1)&(b.feh>=low)&(b.feh<-2)&np.isfinite(b.mh_xp)).to_numpy()
    alt=b.feh.to_numpy().copy();alt[changed]=b.mh_xp.to_numpy()[changed]
    # Direct mass per bin with the alternative and the original [Fe/H]; relative changes below -2, -1.5 and in total
    jj=np.digitize(alt,FEH_EDGES)-1
    m=np.histogram(alt,FEH_EDGES,weights=(b.w*b['fill']).to_numpy()*(1-np.interp(alt,fg.feh_med,fg.f_mw))/e[rid,jj])[0]
    base=np.histogram(b.feh,FEH_EDGES,weights=(b.w*b['fill']).to_numpy()*(1-np.interp(b.feh,fg.feh_med,fg.f_mw))/e[rid,j])[0]
    sens.append(dict(case=label,changed=int(changed.sum()),below_m2=m[0]/base[0]-1,below_m15=m[:3].sum()/base[:3].sum()-1,total=m.sum()/base.sum()-1))
pd.DataFrame(sens).to_csv(OUT/'giant_mass_assignment.csv',index=False)
```

#### Summary numbers
Totals, masses below -2 and -1.5, bootstrap ranges, core ratio, conversion alternatives and definitions, written to
results/giant_mass.json.

```python
# Area of the cores: len(f) spherical caps of radius 0.8 deg, 2 pi (1 - cos r) sr x (180/pi)^2 deg^2/sr.
cum=lambda m,n: float(np.sum(m[:n]))  # noqa: E731
# Summary numbers (masses in Msun formed; below_m2 = first bin, below_m15 = first three bins) and definitions
metadata=dict(fields=len(f),common_area_deg2=float(len(f)*2*np.pi*(1-np.cos(np.deg2rad(.8)))*(180/np.pi)**2),
    mass_type='formed, Kroupa 0.08–100 Msun',full_radius_deg=12,full_mass=float(mass_ref.sum()),covered_mass=float(summary.covered_mass.sum()),
    below_m2=cum(mass_ref,1),below_m15=cum(mass_ref,3),direct_full_mass=float(direct_ref.sum()),direct_below_m2=cum(direct_ref,1),
    direct_below_m15=cum(direct_ref,3),
    below_m2_p16_p84=np.percentile(boot[:,0],[16,84]).tolist(),below_m15_p16_p84=np.percentile(boot[:,:3].sum(1),[16,84]).tolist(),
    full_mass_p16_p84=np.percentile(boot.sum(1),[16,84]).tolist(),
    common_boss_apogee_ratio=float(mb.sum()/ma.sum()),
    common_ratio_p16_p84=np.percentile((draw@mb).sum(axis=1)/(draw@ma).sum(axis=1),[16,84]).tolist(),
    conversion_alternatives={name:dict(full=float(v[(v.conversion==name)].mass.sum()),below_m2=float(v[(v.conversion==name)].mass.iloc[0]),
                                       below_m15=float(v[(v.conversion==name)].mass.iloc[:3].sum()))
                             for name in ['E+','E-','i-','i+']},
    parent_median_ebv=dict(true=float(geo.ebv_true.median()),carton=float(geo.ebv_carton.median())),
    foreground='reference mean fraction over 2-10 deg, radial profile from a uniform foreground surface density per reporting bin; amplitude 0,1,2',
    response='empirical BOSS response from the stars with both spectra (scripts/boss_response.py); per-bin ratio fitted/direct applied to regions',
    envelope='A/B/C age priors x measured/0/0.15/0.3 alpha x 0/1/2 foreground, plus E(V-I) +-0.025 and inclination +-7 deg for the reference; not confidence bounds',
    bootstrap='500 nside16 pixel draws of the parent with selection weights and coverage recomputed, and resampled response overlap, for the full mass; 500 paired field cores for the ratio (weights fixed); conversions fixed',
    grid_age_max_gyr=float(fullgrid.age.max()),grid_mh_min=float(fullgrid.mh.min()),
    input_hashes={x:sha256(DATA/x) for x in ['boss_weights.parquet','apogee_weights.parquet','boss_parent.parquet','apogee_parent.parquet']})
# Write results/giant_mass.json and print it without the input hashes
(OUT/'giant_mass.json').write_text(json.dumps(metadata,indent=2))
print(json.dumps({k:val for k,val in metadata.items() if k!='input_hashes'},indent=2),flush=True)
```

## Present-day mass

The mass formed includes stars that have died. The present-day mass, in living stars and remnants (white dwarfs, neutron stars and
black holes), follows from the same isochrones and age priors: living stars from the current masses along the isochrone, completed
with unevolved stars between 0.08 M☉ and the lowest tabulated mass; remnants from the initial–final mass relations of
[Conroy, Gunn & White (2009)](https://doi.org/10.1088/0004-637X/699/1/486) (section 2), after
[Renzini & Ciotti (1993)](https://doi.org/10.1086/187068), with all remnants retained and returned gas excluded. The present-to-formed
ratio is 0.61 overall and 0.58 below [Fe/H] = −1.5.

### The present-mass computation

`scripts/build_present_mass.py` computes, for every isochrone, the fractions of the formed mass now in living stars and in
remnants, averages them over age with the weights of each conversion, and multiplies the formed masses of
`scripts/build_giant_mass.py` by them. It writes `results/giant_present_mass.csv`, the table of formed and present mass per [Fe/H]
bin that Figure 4.2, Table 4.3 and Letter Fig. 1b read.

<details><summary>About <code>scripts/build_present_mass.py</code></summary>

```text
Add living and present (living+remnant) masses to the formed-mass results: results/giant_present_mass.csv, the stellar
mass formed and present per [Fe/H] bin of Letter Fig. 1b (scripts/paper_letter_signal.py) and of paper Sect. 2.3.

Per isochrone (PARSEC gaiaEDR3 grid, sources.yaml entry parsec_gaiaEDR3), lmcold.surviving_mass.survival_grid gives the
fractions of the formed mass (Kroupa IMF, 0.08-100 Msun) now in living stars and in remnants (Conroy, Gunn & White 2009);
average_survival averages them over age with the same age weights as each formed-mass conversion. The formed masses of
scripts/build_giant_mass.py are multiplied by these fractions.
Inputs (results/): mass_conversion_factors.csv, giant_mass_variants.csv, giant_mass.csv, giant_mass_bootstrap.csv,
giant_mass_fields.csv (all from build_giant_mass.py).
Outputs (results/): mass_survival_grid.csv (fractions per [M/H] and age), mass_survival_factors.csv (conversion rows with the
fractions), giant_present_mass_variants.csv, giant_present_mass.csv (reference: formed, living, remnant and present mass
per reporting bin [Msun], cumulative, bootstrap 16-84% and model min/max), giant_present_mass_fields.csv,
giant_present_mass_common_totals.csv and giant_present_mass.json (totals, below -1.5 and -2, model range, definitions,
input sha256). The survival fractions come from the BOSS 'full' conversion (one per prior x alpha model x [Fe/H] bin) and
are applied to every variant. Used by figures/paper_mass.png, paper_mass_summary.png and Table 3.

Run: .venv/bin/python scripts/build_present_mass.py   (after build_giant_mass.py)
Run time: 2 s on a MacBook (M-series), docs/run_times.txt.
```
</details>

**Data read by `scripts/build_present_mass.py`** (from the run log)

| File | Where it comes from | What it is |
|---|---|---|
| `data/raw/parsec/gaiaEDR3.parquet` | public archive | PARSEC v1.2S + COLIBRI S_LMC_08, Kroupa IMF, gaiaEDR3; earlier download checked against a fresh public slice at [M/H]=-1 |
| `results/giant_mass.csv` | repository | written by `python scripts/build_giant_mass.py` |
| `results/giant_mass_bootstrap.csv` | repository | written by `python scripts/build_giant_mass.py` |
| `results/giant_mass_fields.csv` | repository | written by `python scripts/build_giant_mass.py` |
| `results/giant_mass_variants.csv` | repository | written by `python scripts/build_giant_mass.py` |
| `results/mass_conversion_factors.csv` | repository | written by `python scripts/build_giant_mass.py` |

*Shown, not run here: the tables of scripts/build_giant_mass.py and the PARSEC grid; 2 s. The cells below are the script itself; run it from the repository with* `python scripts/build_present_mass.py`*.*

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

#### Settings
Modules and the output folder.

```python
import json
import numpy as np
import pandas as pd
from lmcold.sources import ROOT,path,sha256
from lmcold.mass_conversion import REFERENCE_PRIOR
from lmcold.surviving_mass import survival_grid,average_survival

out=ROOT/'results'  # folder of the input and output tables
```

#### Load the data and the survival fractions
The PARSEC grid (sources.yaml entry parsec_gaiaEDR3) gives, per isochrone, the fractions of the formed mass now in living
stars and in remnants; they are averaged over age with the weights of each conversion of results/mass_conversion_factors.csv.
The mass tables of build_giant_mass.py are read in the next steps, where they are used.

```python
g=survival_grid(pd.read_parquet(path('parsec_gaiaEDR3')))  # living and remnant fractions of the formed mass per [M/H] and age
g.to_csv(out/'mass_survival_grid.csv',index=False)
f=pd.read_csv(out/'mass_conversion_factors.csv')  # one row per survey x field/region x prior x alpha model x [Fe/H] bin
f=average_survival(g,f)  # adds living_fraction, remnant_fraction, present_fraction (dimensionless)
f.to_csv(out/'mass_survival_factors.csv',index=False)
full=f[(f.survey=='BOSS')&(f.field=='full')]  # fractions of the whole-parent BOSS conversion
```

#### Present mass for every model alternative
Each variant of results/giant_mass_variants.csv times the fractions of the whole-parent BOSS conversion with the same age
weighting and [alpha/Fe] model (results/giant_present_mass_variants.csv).

```python
v=pd.read_csv(out/'giant_mass_variants.csv').merge(full[['prior','alpha_model','feh','living_fraction','present_fraction','present_fraction_legacy_remnants']],on=['prior','alpha_model','feh'],validate='many_to_one')
# 'present_legacy_remnants': an alternative remnant prescription (lmcold.surviving_mass, legacy=True), kept as a sensitivity.
for name,fraction in [('living','living_fraction'),('present','present_fraction'),('present_legacy_remnants','present_fraction_legacy_remnants')]:
    v[name+'_mass']=v.mass*v[fraction]
    v[name+'_cumulative']=v.groupby(['conversion','prior','alpha_model','foreground','filled'])[name+'_mass'].cumsum()
v.to_csv(out/'giant_present_mass_variants.csv',index=False)
```

#### The reference table
Formed, living, remnant and present mass per reporting bin and below each bin, with bootstrap percentiles (fractions fixed)
and model ranges: results/giant_present_mass.csv, the table that Letter Fig. 1b and the mass figures read.

```python
reference=full[(full.prior==REFERENCE_PRIOR)&(full.alpha_model=='measured')]  # AMR age weighting, measured alpha
t=pd.read_csv(out/'giant_mass.csv').merge(reference[['feh','living_fraction','remnant_fraction','present_fraction','present_fraction_legacy_remnants']],on='feh',validate='one_to_one')
t=t.rename(columns={'mass':'formed_mass'})  # Msun formed per reporting bin
boot=pd.read_csv(out/'giant_mass_bootstrap.csv').to_numpy()  # formed mass per bin, pixel and response bootstrap
for name in ['living','present']:
    t[name+'_mass']=t.formed_mass*t[name+'_fraction']
    t[name+'_cumulative']=t[name+'_mass'].cumsum()
    for p in ['p16','p84']:
        t[name+'_mass_'+p]=t['mass_'+p]*t[name+'_fraction']
    draws=boot*t[name+'_fraction'].to_numpy()  # (NBOOT, bins); fractions held fixed
    t[[name+'_cumulative_p16',name+'_cumulative_p84']]=np.percentile(draws.cumsum(axis=1),[16,84],axis=0).T
    # Model range over the filled variants (the rows of each feh group follow the sorted reporting bins).
    for suffix,method in [('min','min'),('max','max')]:
        t[name+'_mass_model_'+suffix]=v[v.filled].groupby('feh')[name+'_mass'].agg(method).to_numpy()
        t[name+'_cumulative_model_'+suffix]=v[v.filled].groupby('feh')[name+'_cumulative'].agg(method).to_numpy()
t['remnant_mass']=t.formed_mass*t.remnant_fraction  # Msun in white dwarfs, neutron stars and black holes
t.to_csv(out/'giant_present_mass.csv',index=False)
```

#### Summary numbers
Totals below [Fe/H] = -1.5, below -2 and overall, the present-to-formed ratio and the model range of the present total.

```python
# Sums [Msun] over the reporting bins: all bins, bins with [Fe/H] < -1.5 and bins with [Fe/H] < -2 (bin centres 'feh').
summary={}
for label,use in [('total',np.ones(len(t),bool)),('below_m15',t.feh<-1.5),('below_m2',t.feh<-2)]:
    q=t[use]
    summary[label]={c:float(q[c].sum()) for c in ['formed_mass','living_mass','remnant_mass','present_mass']}
    # Present-to-formed ratio, and the relative change of the present mass under the alternative remnant prescription.
    summary[label]['present_over_formed']=float(q.present_mass.sum()/q.formed_mass.sum())
    summary[label]['legacy_remnant_mass_change_fraction']=float((q.formed_mass*q.present_fraction_legacy_remnants).sum()/q.present_mass.sum()-1)
# Total present mass [Msun] of every filled model variant; its min and max give the model range.
totals=v[v.filled].groupby(['conversion','prior','alpha_model','foreground']).present_mass.sum()
summary['total_present_model_range']=[float(totals.min()),float(totals.max())]
summary['model_range_definition']='Age/alpha/foreground alternatives and the extinction/inclination conversion alternatives, not a confidence interval; fixed reference remnant prescription.'
```

#### Field cores
Formed, living and present mass per core and survey, each with the fractions of its own conversion, and their sums; the
summary is written to results/giant_present_mass.json.

```python
# Field cores: each survey and core takes its own conversion's fractions.
fields=pd.read_csv(out/'giant_mass_fields.csv').merge(
    f[['survey','field','prior','alpha_model','feh','living_fraction','present_fraction']],
    on=['survey','field','prior','alpha_model','feh'],validate='one_to_one')
# Formed, living and present mass [Msun] per survey x core x prior x alpha model x [Fe/H] bin.
fields['formed_mass']=fields.mass
fields['living_mass']=fields.mass*fields.living_fraction
fields['present_mass']=fields.mass*fields.present_fraction
fields.to_csv(out/'giant_present_mass_fields.csv',index=False)
# Sum over the cores per survey, prior and alpha model; the reference model goes into the JSON summary.
common=fields.groupby(['survey','prior','alpha_model'])[['formed_mass','living_mass','present_mass']].sum().reset_index()
common.to_csv(out/'giant_present_mass_common_totals.csv',index=False)
reference=common.query("prior == @REFERENCE_PRIOR and alpha_model == 'measured'")
summary['common_field_totals']=reference.set_index('survey')[['formed_mass','living_mass','present_mass']].to_dict('index')
# Definitions and the sha256 of the input tables, stored with the numbers.
summary['common_field_definition']='Ten identical non-overlapping 0.8-degree field cores, 20.106 square degrees; velocity-selected, before foreground subtraction. Not full-LMC totals.'
summary['definition']='Present = living stars plus all retained remnants; returned gas excluded; no dynamical loss. Living stars separately tabulated.'
summary['remnants']='Conroy, Gunn & White 2009 section 2; alternative remnant prescription retained as sensitivity, not uncertainty interval.'
summary['input_hashes']={name:sha256(out/name) for name in ['giant_mass.csv','giant_mass_variants.csv','mass_conversion_factors.csv']}
(out/'giant_present_mass.json').write_text(json.dumps(summary,indent=2))
print(json.dumps(summary,indent=2))
```

## Uncertainties

We give two kinds of uncertainty. Statistical errors are the 16th–84th percentiles of 500 bootstrap resamples of
HEALPix ([Górski et al. 2005](https://doi.org/10.1086/427976)) sky pixels at resolution parameter nside = 16 of the parent, with the selection weights and coverage factors recomputed for each resample, and of the stars that
define the response; the conversions are fixed. Model ranges span the age weightings (reference, A, B and C); [α/Fe] from the
measured trend or fixed at 0, 0.15 and 0.3 (capped at 0.1 above [Fe/H] = −1); zero, the reference and
twice the foreground; the systematic error of ±0.025 mag in $E(V-I)$ of Skowron et al. (2021), which changes the total
by ±4 per cent; and the inclination error of ±7°, which changes it by less than
0.3 per cent.

## Result

Figure 4.2 and Table 4.2 give the result (Table 4.3 lists every bin). Within 12° the LMC formed
(2.45 +0.96/−0.96) × 10⁹ M☉ (model range 1.86–3.22
× 10⁹) and holds 1.49 × 10⁹ M☉ today. Of the formed mass, 0.94
(+0.71/−0.40) × 10⁷ M☉ has [Fe/H] < −2 and 5.2
(+2.2/−1.4) × 10⁷ M☉ has [Fe/H] < −1.5: 0.4 and
2.1 per cent of the total; 12 per cent (bootstrap
10–13, model range 9–14 per cent) has
[Fe/H] > −0.5.
Without the response correction the masses below −2 and −1.5 would be 1.8 × 10⁷ and
6.8 × 10⁷ M☉. The mass rises on average by a factor of 2.4 per 0.25 dex from
[Fe/H] = −2 to the peak in the bin -0.75 to -0.5. APOGEE checks this in 10 non-overlapping field cores of
radius 0.8° where both surveys are at least 90 per cent complete: with their own
colour–magnitude selections and conversions, the integrated BOSS/APOGEE mass ratio is 0.91
(+0.02/−0.03) (Figure 4.5).

Table 4.2 and Figure 4.3 compare the totals with published LMC masses, each as published: they differ in
initial mass function, aperture and method, and most give either the formed or the present mass. The formed masses from
colour–magnitude star-formation histories, 2.2 × 10⁹ M☉ ([Harris & Zaritsky 2009](https://doi.org/10.1088/0004-6256/138/5/1243))
and 2.5 × 10⁹ M☉ ([Mazzi et al. 2021](https://doi.org/10.1093/mnras/stab2399)), lie within our bootstrap range;
the published present masses, 1.5–2.0 × 10⁹ M☉
([Kim et al. 1998](https://doi.org/10.1086/306030); [McConnachie 2012](https://doi.org/10.1088/0004-6256/144/1/4);
[Eskew, Zaritsky & Meidt 2012](https://doi.org/10.1088/0004-6256/143/6/139); [Skibba et al. 2012](https://doi.org/10.1088/0004-637X/761/1/42)),
lie within our bootstrap range (0.91–2.08 × 10⁹ M☉) too, except
2.7 × 10⁹ M☉ from integrated light ([van der Marel et al. 2002](https://doi.org/10.1086/343775)), and all are at or
above our reference value; the stellar mass of Kim et al. (1998) refers to the inner 4 kpc and is quoted by
[Keim et al. (2023)](https://doi.org/10.1093/mnras/stad554).

<details><summary>About <code>scripts/paper_mass.py</code></summary>

```text
Paper figures for Sect. 2.3: stellar mass per [Fe/H] (main text) and the survey check in shared field cores (appendix).

Run build_giant_mass.py and build_present_mass.py first. Bands are model alternatives (age priors, alpha, foreground, extinction,
inclination), not probabilities; bars are 16th-84th percentiles of the pixel and response bootstrap with weights and
conversions fixed. The cumulative axis is metallicity, not age or lookback time.
paper_mass.png: (a) mass per bin, (b) mass below [Fe/H]; formed and present (living stars plus remnants) on the same axes.
paper_mass_cores.png: (a) BOSS and APOGEE formed mass per bin in the ten shared field cores, (b) their ratio.

Inputs (results/, from scripts/build_giant_mass.py and scripts/build_present_mass.py):
- giant_present_mass.csv: per reporting [Fe/H] bin (lmcold.mass_samples.FEH_EDGES, open ends), formed_mass, cumulative,
  present_mass, present_cumulative [Msun], each with _p16/_p84 (bootstrap; for formed mass the columns mass_p16/mass_p84) and
  _model_min/_model_max (range over the model alternatives).
- giant_mass_bootstrap.csv: formed mass per bin [Msun] of each bootstrap draw (columns = bin centres).
- giant_mass_common.csv: summed over the ten shared APOGEE/BOSS field cores, boss_mass and apogee_mass [Msun] per bin, their
  ratio with 16th-84th percentiles of a resampling of the cores, and the unweighted star counts boss_raw, apogee_raw.
- giant_mass_fields.csv: mass per field core, survey, age prior, alpha model and bin (for the model ranges of the cores).
- mass_conversion_factors.csv: giants per Msun formed (eta) per bin; the 'full' BOSS conversion with the reference prior and the
  measured alpha trend is used here. mass_alpha_trend.csv: median APOGEE [alpha/M] per [Fe/H] bin.
- data/boss_parent.parquet (release_boss_parent) and data/xp_andrae.parquet (scripts/build_xp_andrae.py, from sources.yaml
  andrae23_table1): Gaia XP [M/H] of the BOSS eligible parent; data/clusters_grocholski06.csv (clusters_grocholski06).
paper_mass.png panels (1 x 3; x: [Fe/H]):
(a) y: mass per reporting bin [Msun] (log), x at the bin centres (ticks '<-2' ... '>=0'); blue: formed, orange: present (living
    stars and remnants); bars: bootstrap 16th-84th percentiles; bands: model range.
(b) y: mass below [Fe/H] [Msun] (log), x at each bin's upper edge, the last point ('All', placed at 0.3) the total; same coding.
(c) y: fraction per dex (log, 5e-3 to 5), x -2.5 to 0.5; steps: our formed mass per bin divided by the total and the bin width
    (outer bins closed at -2.5 and 0.5), band from the bootstrap; purple dashed: Gaia XP [M/H] of the BOSS parent converted to
    [Fe/H] and weighted by 1/eta (i.e. as formed mass); black: Grocholski+06 clusters by number; both Gaussian KDEs with bootstrap
    16th-84th percentile bands.
paper_mass_cores.png panels (1 x 2; x: [Fe/H] reporting bin):
(a) y: formed mass per bin [Msun] (log) in the shared cores, BOSS (blue) and APOGEE (orange), each with its own CMD conversion;
    bands: range over age priors and alpha models.
(b) y: BOSS / APOGEE (log, 0.3-3), core-resampling 16th-84th percentiles, grey band: model range; bins with >= 5 unweighted stars
    in both surveys.
Placement: paper_mass.png Sect. 2.3 main text; paper_mass_cores.png appendix (docs/figures.md).
Run: .venv/bin/python scripts/paper_mass.py
Run time: 10 s on a MacBook (M-series), docs/run_times.txt.
```
</details>

**Data read by `scripts/paper_mass.py`** (from the run log)

| File | Where it comes from | What it is |
|---|---|---|
| `data/boss_parent.parquet` | Zenodo bundle 1 | Gaia DR3 stars within 12 deg of the LMC passing the rough cuts of mwm_magcloud_rgb_boss, with SFD E(B-V) and `eligible` |
| `data/clusters_grocholski06.csv` | public archive | LMC clusters with Ca II triplet [Fe/H] and radial velocities (Grocholski et al. 2006, 2006AJ....132.1630G) |
| `data/xp_andrae.parquet` | Zenodo bundle 1 | Andrae et al. 2023 Gaia XP [M/H] of the selection-function parents |
| `results/giant_mass_bootstrap.csv` | repository | written by `python scripts/build_giant_mass.py` |
| `results/giant_mass_common.csv` | repository | written by `python scripts/build_giant_mass.py` |
| `results/giant_mass_fields.csv` | repository | written by `python scripts/build_giant_mass.py` |
| `results/giant_present_mass.csv` | repository | written by `python scripts/build_present_mass.py` |
| `results/mass_alpha_trend.csv` | repository | written by `python scripts/build_giant_mass.py` |
| `results/mass_conversion_factors.csv` | repository | written by `python scripts/build_giant_mass.py` |

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.ticker import NullFormatter
import numpy as np
from scipy.ndimage import gaussian_filter1d
import pandas as pd
from lmcold.layout import panel_grid
from lmcold.mass_conversion import REFERENCE_PRIOR
from lmcold.mass_samples import FEH_CENTRES, FEH_EDGES
from lmcold.sources import DATA, ROOT

### Load the data
The mass tables written by `scripts/build_giant_mass.py` and `scripts/build_present_mass.py` (`results/giant_present_mass.csv`,
`giant_mass_common.csv`, `giant_mass_fields.csv`, `giant_mass_bootstrap.csv`, `mass_conversion_factors.csv`,
`mass_alpha_trend.csv`); the BOSS parent with its Gaia XP [M/H] (`data/boss_parent.parquet`, `data/xp_andrae.parquet`, Zenodo
bundle 1); and the cluster [Fe/H] of Grocholski et al. (2006), `data/clusters_grocholski06.csv`.

In [ ]:
# results/giant_present_mass.csv: formed and present mass per [Fe/H] bin and below it, bootstrap percentiles and model ranges
t=pd.read_csv(ROOT/'results/giant_present_mass.csv')
# results/giant_mass_common.csv: BOSS and APOGEE formed mass in the ten shared field cores, and their ratio
c=pd.read_csv(ROOT/'results/giant_mass_common.csv')
# results/giant_mass_fields.csv: mass per field core, survey and [Fe/H] bin for each model (prior x alpha)
fields=pd.read_csv(ROOT/'results/giant_mass_fields.csv')
# results/giant_mass_bootstrap.csv: formed mass per [Fe/H] bin in each bootstrap draw (one row per draw)
boot=pd.read_csv(ROOT/'results/giant_mass_bootstrap.csv').to_numpy()
# results/mass_conversion_factors.csv: Msun formed per giant (eta) per bin; the full parent, reference prior, measured alpha
conv=pd.read_csv(ROOT/'results/mass_conversion_factors.csv').query("field=='full' and prior==@REFERENCE_PRIOR and alpha_model=='measured'")
# results/mass_alpha_trend.csv: the APOGEE [alpha/Fe] trend against [Fe/H]
trend=pd.read_csv(ROOT/'results/mass_alpha_trend.csv')
# data/boss_parent.parquet (eligible stars) joined with data/xp_andrae.parquet: XP [M/H] of the BOSS eligible parent (finite
# values only: build_xp_andrae sets NaN outside its cool-giant cuts)
xp=pd.read_parquet(DATA/'boss_parent.parquet',columns=['source_id','eligible']).query('eligible').merge(
    pd.read_parquet(DATA/'xp_andrae.parquet',columns=['source_id','mh_xp']),on='source_id').mh_xp.dropna().to_numpy()
# data/clusters_grocholski06.csv: Ca II triplet [Fe/H] of LMC clusters
cl=pd.read_csv(DATA/'clusters_grocholski06.csv').feh.dropna().to_numpy()

### The mass tables
Formed and present mass per bin with bootstrap percentiles and model ranges, the masses of the shared field cores, and the
model range of the cores over the age and alpha alternatives.

In [ ]:
plt.style.use(ROOT/'src/lmcold/paper.mplstyle')
# Mass summed over the cores for each model (prior x alpha) and survey; model_range: min/max over models of BOSS, APOGEE, ratio.
models=fields.groupby(['prior','alpha_model','survey','feh']).mass.sum().unstack('survey')
models['ratio']=models.BOSS/models.APOGEE
model_range=models.groupby('feh').agg(['min','max'])

### Panels (a) and (b): mass per bin and mass below [Fe/H]

In [ ]:
blue,orange='#2878b5','#d75a20'
fig,axes=panel_grid(1,3,width=7,left=.55,right=.1,hgap=.6,bottom=.5,top=.6)
# (a) mass per bin at the bin centres; (b) cumulative mass at the upper bin edges, the last (open) bin plotted at 0.3 as 'All'.
for ax,col,x in zip(axes[0,:2],['mass','cumulative'],[FEH_CENTRES,np.r_[t.feh_hi.to_numpy()[:-1],.3]]):
    for name,value,color,lab in [('formed','formed_mass' if col=='mass' else 'cumulative',blue,'formed'),
                                 ('present','present_'+col,orange,'present (living stars and remnants)')]:
        pre='' if name=='formed' else 'present_'
        ax.fill_between(x,t[pre+col+'_model_min'],t[pre+col+'_model_max'],color=color,alpha=.18,lw=0)
        ax.vlines(x,t[pre+col+'_p16'],t[pre+col+'_p84'],color=color,lw=.8)
        ax.plot(x,t[value],'-o',ms=3,color=color,label=lab)
    ax.set(yscale='log',xlabel=r'[Fe/H]',xlim=(-2.45,.45))
    ax.set_xticks([-2.25,-1.5,-.5,.25] if col=='mass' else [-2,-1.5,-.5,.3])
    ax.set_xticklabels(['<−2','−1.5','−0.5','≥0'] if col=='mass' else ['−2','−1.5','−0.5','All'])
axes[0,0].set(title='(a) Mass per metallicity bin',ylabel=r'Mass per bin ($M_\odot$)')
axes[0,1].set(title='(b) Mass below [Fe/H]',ylabel=r'Mass below [Fe/H] ($M_\odot$)')

### Panel (c): comparison with Gaia XP and the clusters
Our formed mass per dex, the Gaia XP metallicities of the BOSS parent weighted as mass, and the clusters by number.

In [ ]:
# (c) Distributions per dex of [Fe/H]. Ours: formed mass per bin over the bin width (binned: the response and the split above -1
# work in bins), band from the bootstrap (results/giant_mass_bootstrap.csv). Gaia XP [M/H] of the BOSS parent (Andrae+23; to [Fe/H]
# with the alpha trend), each star weighted by 1/eta of its bin (results/mass_conversion_factors.csv, full parent, reference), and LMC
# clusters with Ca II triplet [Fe/H] (Grocholski+06) by number: Gaussian KDEs (bandwidths KDE_BW), bands from NBOOT_KDE bootstraps.
KDE_BW,NBOOT_KDE=dict(xp=.05,clusters=.1),300  # KDE bandwidths [dex] and number of bootstrap draws of the KDEs
rng=np.random.default_rng(5)
grid=np.linspace(-2.6,.5,311)  # [Fe/H] grid of the KDEs, 0.01-dex step
ax=axes[0,2]
edges=np.r_[-2.5,FEH_EDGES[1:-1],.5]  # reporting bins with the open ends closed at -2.5 and 0.5 for plotting
dens=lambda m: m/m.sum()/np.diff(edges)  # noqa: E731  fraction of the total mass per dex in each bin
lo,hi=np.percentile([dens(b) for b in boot],[16,84],axis=0)
ax.fill_between(edges,np.r_[lo,lo[-1]],np.r_[hi,hi[-1]],step='post',color=blue,alpha=.25,lw=0)
ax.stairs(dens(t.formed_mass.to_numpy()),edges,color=blue,lw=1.5,label='this work, formed mass')


def kde(x,w,bw):
    """Weighted Gaussian KDE on grid, through a 0.01-dex histogram; normalised to unit integral.

    Parameters
    ----------
    x : numpy.ndarray
        [Fe/H] values [dex].
    w : numpy.ndarray
        Weights per value.
    bw : float
        Gaussian standard deviation [dex].

    Returns
    -------
    numpy.ndarray
        Density per dex on `grid`; values outside the grid are dropped before smoothing.
    """
    h=np.histogram(x,np.r_[grid-.005,grid[-1]+.005],weights=w)[0]
    d=gaussian_filter1d(h.astype(float),bw/.01,mode='constant')
    return d/(d.sum()*.01)


# [Fe/H] = [M/H] - log10(0.638 x 10^[alpha/Fe] + 0.362) (Salaris, Chieffi & Straniero 1993; refs.bib Salaris1993); the APOGEE
# alpha trend, tabulated against [Fe/H], is interpolated at [M/H].
feh_xp=xp-np.log10(.638*10**np.interp(xp,trend.feh,trend.alpha)+.362)
w_xp=1/conv.sort_values('feh').eta.to_numpy()[np.digitize(feh_xp,FEH_EDGES)-1]  # Msun formed per giant of the star's bin
for x,w,bw,colour,ls,lab in [(feh_xp,w_xp,KDE_BW['xp'],'#7b3294','--',f'Gaia XP, BOSS parent ({len(xp):,} stars), as mass'),
                         (cl,np.ones(len(cl)),KDE_BW['clusters'],'k','-',f'clusters, Ca II triplet ({len(cl)}), by number')]:
    bs=[kde(x[i],w[i],bw) for i in (rng.integers(0,len(x),len(x)) for _ in range(NBOOT_KDE))]
    ax.fill_between(grid,*np.percentile(bs,[16,84],axis=0),color=colour,alpha=.15,lw=0)
    ax.plot(grid,kde(x,w,bw),color=colour,lw=1.2,ls=ls,label=lab)
ax.set(yscale='log',ylim=(5e-3,5),xlim=(-2.5,.5),xlabel='[Fe/H]',ylabel='fraction per dex',title='(c) Comparison')
ax.legend(loc='lower center',bbox_to_anchor=(.5,1.1),frameon=False,fontsize=6.5)
h,l=axes[0,0].get_legend_handles_labels()
fig.legend(h,l,loc='upper center',ncol=2,bbox_to_anchor=(.35,1))
fig.savefig(ROOT/'figures/paper_mass.png')

In [ ]:
from IPython.display import Image
Image(filename='figures/paper_mass.png')

**Figure 4.2.** Stellar mass of the LMC within 12° per metallicity, from the BOSS giants. (a) Mass per [Fe/H]
bin (M☉, logarithmic) against [Fe/H]; the first bin holds all stars below −2, the last all stars above 0. (b) Mass below each [Fe/H]
(M☉); 'All' is the total. Blue: formed (initial) mass; orange: present mass in living stars and remnants. Points: reference model
(age–metallicity relation, measured [α/Fe], reference foreground); bars: 16th–84th percentiles of the bootstrap; shading: range of the
model alternatives. The cumulative axis orders mass by metallicity, not by time. (c) Distributions per dex of [Fe/H] (fraction of
the total per dex, logarithmic): our formed mass, binned as measured (blue steps; the two open end bins are drawn closed; band: bootstrap); a Gaussian kernel
density (width 0.05 dex) of the *Gaia* XP metallicities
([Andrae, Rix & Chandra 2023](https://doi.org/10.3847/1538-4365/acd53e)) of the whole BOSS parent, converted to [Fe/H] with the [α/Fe]
trend and weighted by the inverse efficiency of each star's bin, i.e. as mass (purple dashed; its bootstrap band is narrower than the
line); and a kernel density (width 0.1 dex) of the LMC clusters with [Fe/H] from the Ca II triplet
([Grocholski et al. 2006](https://doi.org/10.1086/507303)), by number (black; grey: bootstrap over clusters). `scripts/paper_mass.py`

### BOSS against APOGEE in the field cores

In [ ]:
# --- paper_mass_cores.png: BOSS against APOGEE in the ten shared field cores ---
fig,axes=panel_grid(1,2,width=7,left=.6,right=.1,hgap=.65,bottom=.5,top=.7)
for survey,color in [('boss',blue),('apogee',orange)]:
    m=c[survey+'_mass'];valid=m>0  # bins with mass (log axis)
    axes[0,0].fill_between(FEH_CENTRES[valid],model_range[survey.upper()]['min'].to_numpy()[valid],model_range[survey.upper()]['max'].to_numpy()[valid],color=color,alpha=.15)
    axes[0,0].plot(FEH_CENTRES[valid],m[valid],'-o',ms=3,color=color,label=survey.upper())
axes[0,0].set(yscale='log',title='(a) Same field cores, separate CMD conversions',ylabel=r'Formed mass per bin ($M_\odot$)')
valid=(c.boss_raw>=5)&(c.apogee_raw>=5)&np.isfinite(c.ratio_p16)&np.isfinite(c.ratio_p84)
axes[0,1].fill_between(FEH_CENTRES[valid],model_range['ratio']['min'].to_numpy()[valid],model_range['ratio']['max'].to_numpy()[valid],color='0.7',alpha=.3)
axes[0,1].vlines(FEH_CENTRES[valid],c.ratio_p16[valid],c.ratio_p84[valid],color=blue,lw=.8)
axes[0,1].plot(FEH_CENTRES[valid],c.ratio[valid],'o',ms=3,color=blue)
axes[0,1].axhline(1,color='0.5',ls='--',lw=.7)
axes[0,1].set(title='(b) Common-footprint mass ratio',ylabel='BOSS / APOGEE',yscale='log',ylim=(.3,3),yticks=[.5,1,2],yticklabels=['0.5','1','2'])
axes[0,1].yaxis.set_minor_formatter(NullFormatter())
for ax in axes[0]:
    ax.set(xlabel=r'[Fe/H]',xlim=(-2.45,.45),xticks=[-2.25,-1.5,-.5,.25],xticklabels=['<−2','−1.5','−0.5','≥0'])
h,l=axes[0,0].get_legend_handles_labels()
fig.legend(h,l,loc='upper center',ncol=2,bbox_to_anchor=(.5,.99))
fig.savefig(ROOT/'figures/paper_mass_cores.png')
print('Saved paper_mass.png and paper_mass_cores.png')

In [ ]:
from IPython.display import Image
Image(filename='figures/paper_mass_cores.png')

**Figure 4.5.** BOSS and APOGEE in the same 10 field cores. (a) Formed mass per [Fe/H] bin (M☉,
logarithmic) from each survey, BOSS (blue) and APOGEE (orange), each with its own colour–magnitude selection and conversion; bands:
range over the age and [α/Fe] alternatives. (b) BOSS over APOGEE (logarithmic); bars: 16th–84th percentiles of a paired bootstrap of
the cores; grey band: range of the alternatives; dashed: a ratio of 1. Bins need at least 5 stars from each survey. Above
[Fe/H] = −1 the BOSS mass is shared among the bins as the APOGEE giants of the same cell, so there only the total is an independent
test. `scripts/paper_mass.py`

<details><summary>About <code>scripts/paper_mass_summary.py</code></summary>

```text
Paper figure (Sect. 2.3): the total stellar mass of the LMC, this work against published estimates.

One row per study: formed mass (triangles) and present mass in living stars and remnants (squares), as published (no IMF or
aperture conversion; lmcold.constants.LMC_MASS_LITERATURE). This work (results/giant_present_mass.csv): reference value, 16th-84th
bootstrap percentiles (thin bars) and range of the model alternatives (thick light bars), also shaded across all rows.
Writes figures/paper_mass_summary.png.

Input: results/giant_present_mass.csv (scripts/build_giant_mass.py, then scripts/build_present_mass.py), one row per reporting
[Fe/H] bin; its last row (the open bin [Fe/H] >= 0) holds the cumulative mass of all bins, i.e. the total within 12 deg. Columns
used: cumulative and present_cumulative [Msun], with suffixes _p16/_p84 (bootstrap) and _model_min/_model_max (alternatives).
Panel (single): x: stellar mass [10^9 Msun], log, 0.8-4.5; y: one row per study, this work on top (separated by a grey line),
then the published estimates in order of publication year. Blue triangles: formed mass; orange squares: present mass; open
symbols: published values. Paper: Sect. 2.3, main text, with Table 3 (scripts/paper_table_mass_summary.py).
Run: .venv/bin/python scripts/paper_mass_summary.py
Run time: 1 s on a MacBook (M-series), docs/run_times.txt.
```
</details>

**Data read by `scripts/paper_mass_summary.py`** (from the run log)

| File | Where it comes from | What it is |
|---|---|---|
| `results/giant_present_mass.csv` | repository | written by `python scripts/build_present_mass.py` |

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from lmcold.constants import LMC_MASS_LITERATURE
from lmcold.layout import panel_grid
from lmcold.sources import ROOT

### Load the data
The total formed and present mass of this work: the last row of `results/giant_present_mass.csv` (written by
`scripts/build_present_mass.py`). The published masses are not read from a file: they are listed in
`lmcold.constants.LMC_MASS_LITERATURE`.

In [ ]:
# results/giant_present_mass.csv: formed and present mass with bootstrap percentiles and model ranges; last bin = total mass
t = pd.read_csv(ROOT / 'results/giant_present_mass.csv').iloc[-1]  # last bin: cumulative over all [Fe/H] = total mass

### This work and the published masses
The total of this work and the published estimates, ordered by year.

In [ ]:
blue, orange = '#2878b5', '#d75a20'
lit = sorted(LMC_MASS_LITERATURE, key=lambda r: r['label'][-5:-1])  # by the year in the label, e.g. 'Kim et al. (1998)'
labels = ['This work'] + [r['label'] for r in lit]
y = np.arange(len(labels))[::-1]  # row positions, top row (this work) highest

### The figure

In [ ]:
plt.style.use(ROOT / 'src/lmcold/paper.mplstyle')
fig, axes = panel_grid(1, 1, width=3.4, left=1.45, right=.1, bottom=.45, top=.45, aspect=.95)
ax = axes[0, 0]
for pre, colour, mk, dy, lab in [('cumulative', blue, '^', .12, 'formed'), ('present_cumulative', orange, 's', -.12, 'present')]:
    # Reference value, bootstrap 16th/84th percentiles and model range, in 10^9 Msun.
    v, lo, hi, mlo, mhi = (t[pre + s] / 1e9 for s in ('', '_p16', '_p84', '_model_min', '_model_max'))
    ax.axvspan(mlo, mhi, color=colour, alpha=.12, lw=0)  # model range shaded across all rows
    ax.plot([mlo, mhi], [y[0] + dy] * 2, color=colour, lw=4, alpha=.35, solid_capstyle='butt')
    ax.errorbar(v, y[0] + dy, xerr=[[v - lo], [hi - v]], fmt=mk, ms=5, color=colour, lw=1, label=lab)
for yy, r in zip(y[1:], lit):
    formed = r['kind'] == 'formed'  # published as formed mass; otherwise present mass
    ax.plot(r['mass'] / 1e9, yy + (.12 if formed else -.12), '^' if formed else 's', ms=5, mfc='white',
            color=blue if formed else orange)
ax.set(xscale='log', xlim=(.8, 4.5), ylim=(-.7, len(labels) - .3), xlabel=r'Stellar mass ($10^9\,M_\odot$)')
ax.set_xticks([1, 1.5, 2, 3, 4], ['1', '1.5', '2', '3', '4'])
ax.xaxis.set_minor_formatter(matplotlib.ticker.NullFormatter())
ax.set_yticks(y, labels, fontsize=7)
ax.axhline(y[0] - .5, color='.6', lw=.6)
h, l = ax.get_legend_handles_labels()  # present (left) then formed (right), above their columns
ax.legend(h[::-1], l[::-1], loc='lower center', bbox_to_anchor=(.5, 1.0), ncol=2, frameon=False, fontsize=7)
fig.savefig(ROOT / 'figures/paper_mass_summary.png', dpi=250)

In [ ]:
from IPython.display import Image
Image(filename='figures/paper_mass_summary.png')

**Figure 4.3.** Total stellar mass of the LMC ($10^9$ M☉, logarithmic): this work (filled symbols, top row; thin
bars: 16th–84th bootstrap percentiles; thick bars and vertical shading: range of the model alternatives) and published estimates as
published (open symbols, one row per study in order of publication). Blue triangles: formed mass; orange squares: present mass in
living stars and remnants. `scripts/paper_mass_summary.py`

In [ ]:
import numpy as np
import pandas as pd
from lmcold.constants import LMC_MASS_LITERATURE
# Formed and present stellar masses per [Fe/H] bin, cumulative below each edge
t = pd.read_csv('results/giant_present_mass.csv')


def cell(r, key):
    """Reference value (16th–84th bootstrap percentiles) [model range], in units of 10^n Msun."""
    v, lo, hi, mlo, mhi = (r[key + s] for s in ('', '_p16', '_p84', '_model_min', '_model_max'))
    n = int(np.floor(np.log10(v)))
    return f'{v / 10**n:.2f} ({lo / 10**n:.2f}–{hi / 10**n:.2f}) [{mlo / 10**n:.2f}–{mhi / 10**n:.2f}] × 10^{n}'


# This work: all stars (last row of t) and below [Fe/H] = -1, -1.5, -2; then the literature values of lmcold.constants
rows = [('This work, all', cell(t.iloc[-1], 'cumulative'), cell(t.iloc[-1], 'present_cumulative'), 'ρ < 12°, BOSS giants, PARSEC', 'Kroupa')]
rows += [(f'This work, [Fe/H] < {e:g}', cell(t[t.feh_hi == e].iloc[0], 'cumulative'), cell(t[t.feh_hi == e].iloc[0], 'present_cumulative'), '', '')
         for e in (-1, -1.5, -2)]
rows += [(r['label'], f"{r['mass'] / 1e9:.2f} × 10^9" if r['kind'] == 'formed' else '–',
          f"{r['mass'] / 1e9:.2f} × 10^9" if r['kind'] == 'present' else '–', f"{r['coverage']}; {r['method']}", r['imf'])
         for r in LMC_MASS_LITERATURE]
pd.DataFrame(rows, columns=['', 'Formed mass (M☉)', 'Present mass (M☉)', 'Aperture, method', 'IMF']).set_index('')

**Table 4.2.** Stellar mass of the LMC. This work (`results/giant_present_mass.csv`): reference value with the 16th–84th
bootstrap percentiles in parentheses and the range of the model alternatives in brackets, within 12°; rows below the
total give the mass below each [Fe/H]. Published estimates (`lmcold.constants.LMC_MASS_LITERATURE`) are given as published, formed or
present, with their aperture, method and initial mass function.

The comparison panel of Figure 4.2 sets the distribution against two independent ones. The *Gaia* XP metallicities (estimated
from the low-resolution BP/RP spectra of *Gaia*) of the 497 885 parent stars, converted to mass in the same way, give
3.1 per cent of the mass below [Fe/H] = −1.5 (ours 2.1), 0.4 per cent below
−2 (0.4) and 9 per cent above −0.5 (12), with the peak one bin lower, at
-1 to -0.75: XP metallicities of metal-rich giants run low against APOGEE at faint magnitudes, by 0.15–0.17
dex for the stars observed by both at $G$ ≥ 16.5 with APOGEE [Fe/H] from -0.9 to
-0.5. The 28 clusters of Grocholski et al. (2006) are more
metal-rich, with a median of -0.50 and 46 per cent above −0.5: the populous clusters with
spectroscopy are mostly younger than 3 Gyr, and the LMC formed almost no clusters between 4 and 11 Gyr ago
([Piatti 2025](https://doi.org/10.1093/mnras/staf1242)), when much of its field mass formed. The metallicity-structure chapter compares
our giants, by number, with the published field-star distributions in the same places: the Ca II triplet distributions of the bar
([Cole et al. 2005](https://doi.org/10.1086/428007)) and of fields 5–8° north of it ([Carrera et al. 2008](https://doi.org/10.1088/0004-6256/135/3/836))
lie 0.04–0.13 dex above BOSS, less than the 0.2 dex by which the Ca II
triplet scale exceeds ours for the stars in common (79 with Carrera et al. 2008, 3 with
Cole et al. 2005), and the *Gaia* XP Strömgren metallicities of [Omkumar et al. (2025)](https://doi.org/10.1051/0004-6361/202452510)
have the BOSS median but a much broader distribution.

## Checks of the mass

In [ ]:
import pandas as pd
# Formed and present stellar masses per [Fe/H] bin, cumulative below each edge
t = pd.read_csv('results/giant_present_mass.csv')


def cell(r, key, pre=None):
    """Value r[key] (16th–84th bootstrap percentiles) [model range] in 10^6 Msun; the percentile and range columns start with pre."""
    v = r[key] / 1e6
    lo, hi, mlo, mhi = (r[(pre or key) + s] / 1e6 for s in ('_p16', '_p84', '_model_min', '_model_max'))
    f = lambda x: f'{x:.3g}' if x < 1000 else f'{x:.0f}'  # noqa: E731, three significant figures
    return f'{f(v)} ({f(lo)}–{f(hi)}) [{f(mlo)}–{f(mhi)}]'


# Row labels: the [Fe/H] bins; columns: mass in each bin and below its upper edge, formed and present
lab = ['< −2'] + [f'{lo:g} to {hi:g}'.replace('-', '−') for lo, hi in zip(t.feh_lo[1:-1], t.feh_hi[1:-1])] + ['≥ 0']
pd.DataFrame({'Formed': [cell(r, 'formed_mass', 'mass') if r.formed_mass > 0 else '0' for _, r in t.iterrows()],
              'Present': [cell(r, 'present_mass') if r.present_mass > 0 else '0' for _, r in t.iterrows()],
              'Below': [f'{x:g}' for x in t.feh_hi[:-1]] + ['all'],
              'Formed below': [cell(r, 'cumulative') for _, r in t.iterrows()],
              'Present below': [cell(r, 'present_cumulative') for _, r in t.iterrows()]}, index=lab)

**Table 4.3.** Formed and present stellar mass of the LMC within 12°, in $10^6$ M☉, per [Fe/H] bin and below the
upper edge of each bin ('Below'). Values: reference model with the 16th–84th bootstrap percentiles in parentheses; brackets: range of
the model alternatives. Above [Fe/H] = −1 the BOSS mass is shared among the bins as the APOGEE giants of the same cell; no APOGEE giant
of these cells lies above 0 (`results/giant_present_mass.csv`, written by `scripts/build_giant_mass.py` and
`scripts/build_present_mass.py`).

Table 4.3 lists the formed and present mass of every [Fe/H] bin, with bootstrap percentiles and model ranges.

<details><summary>About <code>scripts/paper_mass_method.py</code></summary>

```text
Paper figure (Sect. 2.3): the two steps from giant counts to mass.

(a) Observable BOSS giants per formed mass against age at four model metallicities, one per prior range of [Fe/H]; thick bars:
16th-84th percentiles of the reference age weighting (age-metallicity relation of Povick+24, lmcold.mass_conversion.age_weights)
for the reporting [Fe/H] bin of each curve; thin bars: the window of prior A. The table also holds a representative
APOGEE selection (H<13.3, maglim=14.5; not an average over the field depths), for the appendix comparison.
results/age_weighting.json: mean and 16th-84th percentile ages of the reference weighting per prior [Fe/H] range (Table 2).
(b) BOSS metallicity response: corrected BOSS counts in the common parent divided by the corrected APOGEE counts, and by the
APOGEE counts passed through the response measured on the stars with both spectra (results/boss_response.json from
scripts/boss_response.py); bars are 16th-84th percentiles of the joint pixel and overlap-star bootstrap. Uses the treatment of isolated age-grid spikes of lmcold.isochrones, retaining both raw and adjusted
values in the output table. Photometric distance and depth from constants; the median true reddening of the BOSS parent
(lmcold.extinction) and the median reddening the carton removed; no disc distance offset.

Inputs:
- data/boss_parent.parquet (sources.yaml release_boss_parent): the BOSS eligible parent, for the median reddenings.
- PARSEC isochrones (sources.yaml parsec_gaiaEDR3 for BOSS, parsec_2mass_spitzer for APOGEE; read through lmcold.sources.path,
  which checks the sha256).
- results/mass_alpha_trend.csv (scripts/build_giant_mass.py): median APOGEE [alpha/M] (ASPCAP ALPHA_M) per [Fe/H] bin.
- results/boss_response.json (scripts/boss_response.py): key closure.all with the observed and predicted BOSS counts, the
  direct APOGEE counts and the bootstrap ratio percentiles per reporting [Fe/H] bin (lmcold.mass_samples.FEH_CENTRES).
Method: lmcold.isochrones.count_grid integrates the Kroupa IMF along each isochrone inside the survey's colour-magnitude
selection (BOSS: the mwm_magcloud_rgb_boss polygon and G <= 17.5; APOGEE: its targeting polygon to H = 13.3), at the
photometric LMC distance modulus with a 0.1-mag Gaussian depth (lmcold.constants), giving giants per solar mass formed (n_box);
smooth_age_spikes replaces isolated spikes of the age grid (raw values kept in n_box_raw).
Outputs:
- results/giant_conversion.csv: per survey, isochrone [M/H] (mh) and age: logage, age [Gyr], n_box_raw and n_box [giants per Msun
  formed], spike_adjusted, and the reddenings used (ebv, ebv_carton [mag, SFD scale]).
- results/age_weighting.json: per prior [Fe/H] range, mean age and 16th-84th percentile ages [Gyr] of the reference weighting.
- figures/paper_mass_method.png (appendix D; docs/figures.md).
Panels (1 x 2):
(a) x: age [Gyr], log, 0.4-13.5; y: BOSS giants per 10^6 Msun formed (log, 3-3000); one curve per isochrone [M/H] (-2.0, -1.0,
    -0.6, -0.2 nearest grid values), labelled with the corresponding [Fe/H]; thick horizontal bars: 16th-84th percentile ages of
    the reference (AMR) weighting, thin bars: the age window of prior A (bar heights arbitrary, stacked by curve).
(b) x: [Fe/H] reporting bin centre (-2.45 to -0.1); y: observed BOSS / expected, 0-2.6; grey: corrected BOSS / corrected APOGEE
    counts (no response); blue: corrected BOSS / APOGEE counts passed through the BOSS response, with bootstrap 16th-84th
    percentiles; bins with >= 3 weighted APOGEE giants.
Run: .venv/bin/python scripts/paper_mass_method.py
Run time: 3 s on a MacBook (M-series), docs/run_times.txt.
```
</details>

**Data read by `scripts/paper_mass_method.py`** (from the run log)

| File | Where it comes from | What it is |
|---|---|---|
| `data/boss_parent.parquet` | Zenodo bundle 1 | Gaia DR3 stars within 12 deg of the LMC passing the rough cuts of mwm_magcloud_rgb_boss, with SFD E(B-V) and `eligible` |
| `data/raw/parsec/2mass_spitzer.parquet` | public archive | PARSEC v1.2S + COLIBRI S_LMC_08, Kroupa IMF, 2mass_spitzer; earlier download checked against a fresh public slice at [M/H]=-1 |
| `data/raw/parsec/gaiaEDR3.parquet` | public archive | PARSEC v1.2S + COLIBRI S_LMC_08, Kroupa IMF, gaiaEDR3; earlier download checked against a fresh public slice at [M/H]=-1 |
| `data/raw/skowron21/table1.dat.gz` | public archive | OGLE-IV red-clump E(V-I) map of the Magellanic Clouds, 81,684 boxes of 3.4-27.5 arcmin; Skowron et al. 2021, ApJS 252, 23 (2021ApJS..252...23S, doi:10.3847/1538-4365/abcb81) |
| `results/boss_response.json` | repository | written by `python scripts/boss_response.py` |
| `results/mass_alpha_trend.csv` | repository | written by `python scripts/build_giant_mass.py` |

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import json

from lmcold.isochrones import count_grid, smooth_age_spikes
from lmcold.mass_conversion import AGE_RANGES, REFERENCE_PRIOR, age_weights
from lmcold.mass_samples import FEH_CENTRES
from lmcold.layout import panel_grid
from lmcold.sources import ROOT, DATA, path
from lmcold import magcloud
from lmcold.constants import LMC_TARGET_MS, SMC_TARGET_MS
from lmcold.extinction import rc_map, true_ebv

### Load the data
The BOSS parent `data/boss_parent.parquet` (Zenodo bundle 1), the APOGEE [alpha/Fe] trend `results/mass_alpha_trend.csv`
(written by `scripts/build_giant_mass.py`) and the closure test of `results/boss_response.json` (`scripts/boss_response.py`).
Two further inputs are read where they are used: the Skowron et al. (2021) red-clump reddening map by
`lmcold.extinction.rc_map()` in the reddening step, and the two PARSEC isochrone grids (`lmcold.sources.path`) one per survey
inside the loop that counts giants per solar mass.

In [ ]:
# data/boss_parent.parquet: the BOSS parent (eligible stars kept) with ra, dec [deg] and SFD E(B-V) (column ebv)
p = pd.read_parquet(DATA/'boss_parent.parquet').query('eligible')
# results/mass_alpha_trend.csv: the APOGEE [alpha/Fe] trend against [Fe/H]
trend=pd.read_csv(ROOT/'results/mass_alpha_trend.csv')
# results/boss_response.json: closure test on the common parent, all stars (observed, predicted, APOGEE counts per bin)
c=json.loads((ROOT/'results/boss_response.json').read_text())['closure']['all']

### Reddening of the parent
The median reddening that the carton removed and the median true reddening of the BOSS parent.

In [ ]:
# Median carton (effective) and true reddening of the BOSS eligible parent. The carton's E(B-V) is SFD98 times 0.07 where
# E(B-V) >= 0.1 within 4.5 deg of the LMC or 2.5 deg of the SMC target centres in Magellanic Stream coordinates (the rule of
# lmcold.magcloud.rescaled_ebv, written out here); the true E(B-V) (lmcold.extinction.true_ebv) is the Skowron et al. (2021)
# red-clump E(V-I) / 1.237 inside their map and SFD outside it.
L,B=magcloud.ms_coordinates(p.ra.to_numpy(),p.dec.to_numpy(),np.zeros(len(p)),np.zeros(len(p)))[:2]
effective=np.where(((np.hypot(L-LMC_TARGET_MS[0],B-LMC_TARGET_MS[1])<=4.5)|(np.hypot(L-SMC_TARGET_MS[0],B-SMC_TARGET_MS[1])<=2.5))&(p.ebv.to_numpy()>=.1),.07*p.ebv,p.ebv)
ebv_carton=float(np.median(effective))
ebv=float(np.median(true_ebv(p.ra.to_numpy(),p.dec.to_numpy(),p.ebv.to_numpy(),rc_map())))

### Giants per solar mass formed
For the four isochrone metallicities of panel (a), the number of selected giants per solar mass formed at each age
(`lmcold.isochrones.count_grid`), for BOSS and APOGEE; written to `results/giant_conversion.csv`.

In [ ]:
parts=[]
for survey,key in [('BOSS','parsec_gaiaEDR3'),('APOGEE','parsec_2mass_spitzer')]:
    g=pd.read_parquet(path(key))
    mh=np.array(sorted(g.MH.unique()))
    # The isochrone metallicities nearest to [M/H] = -2, -1, -0.6, -0.2: one per prior [Fe/H] range.
    chosen=[mh[np.argmin(abs(mh-v))] for v in [-2.,-1.,-.6,-.2]]  # [Fe/H] bins <-1.5, -1.5..-1, -1..-0.5, >-0.5
    # Giants per Msun formed for each (mh, age); dmu = 0: no inclined-disc distance offset.
    t=smooth_age_spikes(count_grid(g[g.MH.isin(chosen)],survey,ebv,ebv_carton,0.))
    t['survey']=survey;t['ebv']=ebv;t['ebv_carton']=ebv_carton
    parts.append(t)
t=pd.concat(parts,ignore_index=True)
t.to_csv(ROOT/'results/giant_conversion.csv',index=False)

### Ages of the reference weighting
The 16th-84th percentile and mean ages of the formed mass per [Fe/H] range (the table of ages), written to
`results/age_weighting.json`.

In [ ]:
# Ages of the reference weighting per prior [Fe/H] range (Table 2): the sum of the reporting-bin weights over the range.
ages=np.sort(t.age.unique())
ranges={}
for name,(lo,hi) in {'<-1.5':(-9,-1.5),'-1.5..-1':(-1.5,-1.),'-1..-0.5':(-1.,-.5),'>-0.5':(-.5,9)}.items():
    w=sum(age_weights(ages,f,REFERENCE_PRIOR) for f in FEH_CENTRES if lo<f<hi)  # formed-mass weight per grid age
    # Percentile ages: linear interpolation of the cumulative weight over the sorted grid ages.
    ranges[name]=dict(mean=float(np.average(ages,weights=w)),p16_p84=np.interp([.16,.84],np.cumsum(w)/w.sum(),ages).tolist())
(ROOT/'results/age_weighting.json').write_text(json.dumps(ranges,indent=1))
print(json.dumps(ranges,indent=1))

### Panel (a): efficiency against age

In [ ]:
plt.style.use(ROOT/'src/lmcold/paper.mplstyle')
fig,axes=panel_grid(1,2,width=7,left=.6,right=.1,hgap=.65,bottom=.5,top=.75)
colors=['#d75a20','#33976b','#2878b5','#7b3294']
ax=axes[0,0]
# [M/H] -> [Fe/H] by inverting [M/H] = [Fe/H] + log10(0.638 x 10^[alpha/Fe] + 0.362) (Salaris, Chieffi & Straniero 1993; refs.bib
# Salaris1993), with [alpha/Fe] from the measured APOGEE trend; the trend is interpolated at [M/H] rather than at [Fe/H].
mh_to_feh=lambda mh: mh-np.log10(.638*10**np.interp(mh,trend.feh,trend.alpha)+.362)  # noqa: E731, inverse of the alpha correction
for k,((mh,q),color) in enumerate(zip(t[t.survey=='BOSS'].groupby('mh'),colors)):
    feh=mh_to_feh(mh)
    w=age_weights(q.age.to_numpy(),feh,REFERENCE_PRIOR)  # reference (AMR) weighting of the reporting bin containing feh
    lo,hi=np.interp([.16,.84],np.cumsum(w)/w.sum(),q.age)
    alo,ahi=AGE_RANGES['A'][np.digitize(feh,[-1.5,-1.,-.5])]  # prior-A age window [Gyr] of the curve's [Fe/H] range
    ax.plot(q.age,q.n_box*1e6,color=color,label=rf'[M/H]$={mh:g}$ ([Fe/H]$\approx{feh:.1f}$)')
    # Bars at arbitrary heights, stepped by a factor 1.45 per curve; the prior-A bar just below the AMR bar.
    ax.plot([lo,hi],[600*1.45**k]*2,color=color,lw=2.5,solid_capstyle='butt')
    ax.plot([alo,min(ahi,q.age.max())],[600*1.45**k/1.12]*2,color=color,lw=.8,solid_capstyle='butt')
ax.set(xscale='log',yscale='log',xlim=(.4,13.5),ylim=(3,3000),xticks=[.5,1,2,5,10],xticklabels=['0.5','1','2','5','10'],
       xlabel='Age (Gyr)',ylabel=r'BOSS giants per $10^6\,M_\odot$ formed',title='(a) Conversion')
ax.legend(loc='lower left',bbox_to_anchor=(0,1.1),borderaxespad=0)

### Panel (b): closure of the response on the common parent
From `results/boss_response.json` (`scripts/boss_response.py`).

In [ ]:
ax=axes[0,1]
obs,pred,apo=(np.array(c[k]) for k in ('observed','predicted','apogee'))
good=apo>=3  # bins with at least 3 (weighted) APOGEE giants in the common parent
ax.plot(FEH_CENTRES[good]-.03,obs[good]/apo[good],'o',ms=3,color='0.6',label='no response')
ax.vlines(FEH_CENTRES[good]+.03,np.array(c['ratio_p16'])[good],np.array(c['ratio_p84'])[good],color='#2878b5',lw=.8)
ax.plot(FEH_CENTRES[good]+.03,obs[good]/pred[good],'o',ms=3,color='#2878b5',label='with response')
ax.axhline(1,color='.3',lw=.6,ls='--')
ax.set(xlim=(-2.45,-.1),ylim=(0,2.6),xticks=[-2.25,-1.5,-1,-.5],xticklabels=['<−2','−1.5','−1','−0.5'],xlabel='[Fe/H]',
       ylabel='BOSS observed / expected',title='(b) BOSS response, common parent')
ax.legend(loc='lower left',bbox_to_anchor=(0,1.1),borderaxespad=0)
fig.savefig(ROOT/'figures/paper_mass_method.png')
print(t.groupby(['survey','mh']).n_box_raw.agg(['min','max']))

In [ ]:
from IPython.display import Image
Image(filename='figures/paper_mass_method.png')

**Figure 4.4.** From giant counts to mass. (a) BOSS giants per $10^6$ M☉ formed, $\eta$ (logarithmic), against age
(Gyr, logarithmic) for a single-age population at four model metallicities (legend; orange, green, blue and purple from metal-poor to
metal-rich), one in each [Fe/H] range of Table 4.1. Each curve takes the PARSEC v1.2S isochrone with COLIBRI TP-AGB tracks, from the
main sequence to the end of the AGB; places it at distance modulus 18.477 with a Gaussian depth of 0.1
mag; reddens it by the median true $E(B-V)$ = 0.083 of the parent and de-reddens it by the median
0.038 that the carton removed; and counts the stars inside the carton's colour–magnitude polygon with observed
$G\leq$ 17.5, integrating the Kroupa (2001) IMF, normalised to 1 M☉ formed between 0.08 and
100 M☉, along the isochrone in 40 steps between tabulated points. The curves have no disc-distance
offset; the masses use the regional values. Horizontal bars (placed at arbitrary heights, in the colour of their curve) show the ages
over which $\eta$ is averaged for the 0.25-dex [Fe/H] bin that contains each curve's metallicity. Thick bars: the
reference weighting, i.e. the distribution of formed mass in age expected for that bin from the LMC age–metallicity relation of
Povick et al. (2024) with a constant star-formation rate; the bar spans its 16th–84th percentiles. Thin bars: the fixed window A of
Table 4.1, an alternative. (b) The BOSS metallicity response, tested on the parent common to BOSS and APOGEE: corrected BOSS counts
divided by corrected APOGEE counts (grey) and by the APOGEE counts passed through the response measured on the stars observed by both
(blue), against [Fe/H]; dashed: a ratio of 1. Bars: 16th–84th percentiles of 500 joint bootstraps of sky pixels
and of the stars observed by both. Above [Fe/H] = −1 the masses do not use this response: the BOSS mass there is shared as the APOGEE
giants of the same cell. `scripts/paper_mass_method.py`

**The BOSS response.** Figure 4.1(a) shows the stars observed by both surveys: the calibrated BOSS [Fe/H] is low by
0.15–0.17 dex only between [Fe/H]<sub>APOGEE</sub> = −2 and −1.5. Figure 4.1(b)
repeats the closure test of Figure 4.4(b) by APOGEE field type. In LMC18–35, BOSS holds 0.91
(+0.20/−0.20) of the stars predicted below [Fe/H] = −1.5; in LMC1–17 it holds
0.55 (+0.38/−0.18). APOGEE thus has more metal-poor stars than BOSS
where it used its unpublished giant flag from Washington and DDO51 photometry, as the test of the APOGEE selection against *Gaia* XP
metallicities in chapter 2 also suggests.

**BOSS against APOGEE.** Figure 4.5 compares the masses of the two surveys in 10 non-overlapping
field cores of radius 0.8° (20.1 deg² in all), where APOGEE covers at least
90 per cent of its candidates to $H$ = 13.3 and BOSS at least 90 per cent of
its parent. Each survey uses its own colour–magnitude selection and the extinction and disc offset of each core. The integrated ratio
is 0.91 (+0.02/−0.03). The bins differ more: below [Fe/H] = −2 BOSS is
0.85 (+0.66/−0.24) times APOGEE, from 5 APOGEE stars,
and at -1.75 to -1.5 it is 0.48 (+0.24/−0.14), where
6 of the 10 cores are LMC1–17 fields, targeted with the Washington + DDO51 giant flag. Above −1
the total agrees, 0.97.

**Dust.** The conversion assumes that the *Gaia* parent loses stars to extinction only as the isochrones, reddened with the median
$E(V-I)$ of each region, predict. Figure 4.6 tests this within 4.5° of the centre, where the
red-clump map varies on scales of a few arcmin. In 3.4-arcmin cells (HEALPix pixels) we count the parent stars,
$N_G$, and the 2MASS giants inside the APOGEE target polygon with 12.35 ≤ $H$ ≤ 13.3, $N_H$, which
extinction affects much less ($A_H/A_G$ = 0.20, where $A_H$ and $A_G$ are the extinctions in $H$ and $G$). The
isochrones predict the ratio $f=N_G/N_H$ as a function of reddening for a population of ages 2–12
Gyr and [M/H] = -1.0 to -0.4; the observed $\sum N_G/\sum (fN_H)$ is normalised to the cells with
$E(V-I)$ < 0.10. With $f$ at the regional median, as in the conversion, the parent holds 0.99 of the
predicted stars at $E(V-I)$ = 0.10–0.15, 0.94 at 0.15–0.30,
0.87 ± 0.04 at 0.30–0.40 and 0.67 ± 0.08 at
0.40–0.50. Cells above $E(V-I)$ = 0.15 hold 16.5 per cent of the
central parent, and the missing stars amount to 1.1 per cent of it, a deficit well inside the
±0.025 mag extinction term of the model range. With $f$ at the $E(V-I)$ of each cell the prediction overshoots the loss by
up to 30 per cent: giants in front of the dust are barely dimmed, so the mean red-clump reddening of a dusty
sightline overstates the extinction of its giants. We therefore keep the regional medians. The test assumes that 2MASS is equally
complete in all cells and that the population does not change with reddening; crowding in the bar would hide part of the loss.

<details><summary>About <code>scripts/paper_parent_extinction.py</code></summary>

```text
Does the BOSS parent lose stars to dust as the mass conversion assumes? Infrared test in the central LMC against 2MASS.

Cells are HEALPix nside=1024 pixels (3.4 arcmin, the finest box of the Skowron et al. 2021 map) within 4.5 deg of the centre and
inside the map (SFD saturates in the LMC, so no fallback):
  N_G: parent stars of the re-implemented carton (data/boss_parent.parquet, eligible);
  N_H: 2MASS stars in the APOGEE giant polygon with 12.35 <= H <= 13.3 (data/twomass_centre.parquet); A_H = 0.20 A_G.
The model ratio f = n_G / n_H per Msun formed is from lmcold.isochrones.count_grid for a fixed population (ages 2-12 Gyr equal
mass, [M/H] = -1.0, -0.7, -0.4) on a grid of true and carton E(B-V). Each cell is predicted twice:
  'cell':   f at the cell's own reddening (Skowron E(V-I), its SFD-based carton reddening);
  'region': f at the median reddening of its conversion region (results/mass_regions.csv), as build_giant_mass.py does.
In bins of cell E(V-I), q = sum(N_G) / sum(N_H f), normalised to cells with E(V-I) < 0.1. A complete parent with the right dust model
gives q_cell = 1 at all E(V-I). Errors: bootstrap over cells. Per region, sum(N_H f_cell) / sum(N_H f_region) is the parent fraction
the cell-by-cell model expects relative to the regional median: the regional mass is high by its inverse.
Run build_giant_mass.py first. Writes figures/paper_parent_extinction.png, results/parent_extinction.csv (per E(V-I) bin) and data/parent_extinction_cells.parquet.

Inputs: data/boss_parent.parquet (release_boss_parent; eligible stars, SFD E(B-V) per star), data/twomass_centre.parquet
(sources.yaml twomass_centre), Skowron et al. (2021) map (skowron21_table1 via lmcold.extinction.rc_map), PARSEC isochrones
(parsec_gaiaEDR3, parsec_2mass_spitzer), results/mass_regions.csv (scripts/build_giant_mass.py: median true and carton E(B-V)
of each of the 16 conversion regions, index 4 x ring + quadrant).
Model population: every isochrone of the grid with [M/H] in MH and age 2-12 Gyr counts once (the grid is uniform in log age,
step 0.05 dex), i.e. n_box_raw summed over those isochrones.
Outputs:
- data/parent_extinction_cells.parquet, one row per cell: pix (HEALPix nside 1024, ring order), n_g, n_h, ra, dec [deg],
  region, evi [mag], f_cell, f_region.
- results/parent_extinction.csv, one row per model (cell, region) and E(V-I) bin: evi_lo, evi_hi, n_cells, n_g, q, q_p16, q_p84.
Panels (1 x 2):
(a) x (east, increasing to the left), y (north) [deg] about the LMC centre, +-4.5 deg; cells coloured by Skowron E(V-I) [mag],
    0-0.4, colour bar above.
(b) x: E(V-I) of the cell [mag], 0-0.6 (bin centres, offset +-0.004 by model); y: q (normalised observed / predicted parent
    count), 0-1.6; blue: model at each cell's reddening, orange: model at the region median; bars: bootstrap 16th-84th
    percentiles; bins with >= 50 parent stars.
Placement: appendix (Sect. 2.3; docs/figures.md). Run: .venv/bin/python scripts/paper_parent_extinction.py
Run time: 27 s on a MacBook (M-series), docs/run_times.txt.
```
</details>

**Data read by `scripts/paper_parent_extinction.py`** (from the run log)

| File | Where it comes from | What it is |
|---|---|---|
| `data/boss_parent.parquet` | Zenodo bundle 1 | Gaia DR3 stars within 12 deg of the LMC passing the rough cuts of mwm_magcloud_rgb_boss, with SFD E(B-V) and `eligible` |
| `data/raw/parsec/2mass_spitzer.parquet` | public archive | PARSEC v1.2S + COLIBRI S_LMC_08, Kroupa IMF, 2mass_spitzer; earlier download checked against a fresh public slice at [M/H]=-1 |
| `data/raw/parsec/gaiaEDR3.parquet` | public archive | PARSEC v1.2S + COLIBRI S_LMC_08, Kroupa IMF, gaiaEDR3; earlier download checked against a fresh public slice at [M/H]=-1 |
| `data/raw/skowron21/table1.dat.gz` | public archive | OGLE-IV red-clump E(V-I) map of the Magellanic Clouds, 81,684 boxes of 3.4-27.5 arcmin; Skowron et al. 2021, ApJS 252, 23 (2021ApJS..252...23S, doi:10.3847/1538-4365/abcb81) |
| `data/twomass_centre.parquet` | public archive | 2MASS Point Source Catalog (Skrutskie et al. 2006, 2006AJ....131.1163S), 12 <= H <= 14 within 5 deg of the LMC centre |
| `results/mass_regions.csv` | repository | written by `python scripts/build_giant_mass.py` |

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

In [ ]:
import healpy as hp
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.interpolate import RegularGridInterpolator
from scipy.spatial import cKDTree

from lmcold import apogee_target, magcloud
from lmcold.constants import LMC
from lmcold.extinction import EVI_PER_EBV, rc_map, true_ebv
from lmcold.footprint import angular_distance, lmc_xy
from lmcold.isochrones import count_grid
from lmcold.layout import panel_grid
from lmcold.sources import DATA, ROOT, path

### Settings and the model ratio
Cells, tested area, the fixed isochrone population, and the model ratio f = n_G / n_H as a function of the true and the
carton reddening.

In [ ]:
NSIDE, RMAX = 1024, 4.5  # HEALPix resolution of the cells; radius [deg] of the tested area
RINGS = [0, 3, 6, 9, 12]  # deg, ring edges of the conversion regions (as in build_giant_mass.py)
EBV = np.arange(0, 1.01, .025)  # mag, grid of true E(B-V) for the model ratio
EBV_CARTON = np.array([0, .01, .02, .03, .05, .08, .12, .2, .35, .6, 1.])  # mag, grid of the carton's E(B-V)
MH, AGES = [-1.0, -0.7, -0.4], (2., 12.)  # isochrone [M/H] and age range [Gyr] of the fixed model population
EVI_EDGES = np.array([0, .05, .1, .15, .2, .25, .3, .4, .5, .8])  # mag, bins of cell E(V-I)


def model_ratio():
    """f(E(B-V), carton E(B-V)) = parent-box / H-box giants per Msun, summed over the fixed population.

    Returns
    -------
    scipy.interpolate.RegularGridInterpolator
        Linear interpolator of the ratio on the (EBV, EBV_CARTON) grid [mag]; called with an (N, 2) array of (true E(B-V),
        carton E(B-V)). The 2MASS count depends on the true E(B-V) only (A_H, E(J-Ks) from count_grid).
    """
    def pick(g):
        """Rows of isochrone grid g with [M/H] in MH and age (from logAge, Gyr) within AGES."""
        age = 10**g.logAge / 1e9
        return g[np.isin(g.MH.round(2), MH) & (age >= AGES[0]) & (age <= AGES[1])]
    # PARSEC grids of the fixed population: Gaia EDR3 bands (BOSS parent box) and 2MASS bands (H box).
    boss, twomass = pick(pd.read_parquet(path('parsec_gaiaEDR3'))), pick(pd.read_parquet(path('parsec_2mass_spitzer')))
    # Giants per Msun in the H box at each true E(B-V), shape (len(EBV),); in the parent box on the (EBV, EBV_CARTON) grid.
    h = np.array([count_grid(twomass, 'APOGEE', e, e, 0.).n_box_raw.sum() for e in EBV])
    g = np.array([[count_grid(boss, 'BOSS', e, c, 0.).n_box_raw.sum() for c in EBV_CARTON] for e in EBV])
    return RegularGridInterpolator((EBV, EBV_CARTON), g / h[:, None])


def xyz(ra, dec):
    """Unit vectors, shape (N, 3), of positions ra, dec [deg]."""
    r, d = np.radians(ra), np.radians(dec)
    return np.column_stack([np.cos(d) * np.cos(r), np.cos(d) * np.sin(r), np.sin(d)])

### Load the data
The BOSS parent `data/boss_parent.parquet`, the 2MASS stars of the LMC centre `data/twomass_centre.parquet`, and the median
reddening of each conversion region from `results/mass_regions.csv` (written by `scripts/build_giant_mass.py`). Two further
inputs are read where they are used: the PARSEC isochrone grids inside `model_ratio()` above, and the Skowron et al. (2021)
red-clump reddening map by `lmcold.extinction.rc_map()` in the next cell.

In [ ]:
# data/boss_parent.parquet: the BOSS parent (eligible stars kept) with ra, dec [deg] and SFD E(B-V) (column ebv)
p = pd.read_parquet(DATA / 'boss_parent.parquet').query('eligible')
# data/twomass_centre.parquet: 2MASS stars of the LMC centre with J, H, Ks magnitudes (j_m, h_m, k_m)
t = pd.read_parquet(DATA / 'twomass_centre.parquet')
# results/mass_regions.csv: per conversion region the median true and carton E(B-V)
reg = pd.read_csv(ROOT / 'results/mass_regions.csv', index_col='region')

### Counts per cell
Parent stars and 2MASS giants per cell, the conversion region and reddening of each cell, and the model ratio at the
cell's reddening and at its region's median.

In [ ]:
# 2MASS stars in the APOGEE giant colour-magnitude polygon (maglim 14.5) with H <= 13.3.
t = t[(apogee_target.rgb_class((t.j_m - t.k_m).to_numpy(), t.h_m.to_numpy(), 14.5) > 0) & (t.h_m <= 13.3)]
# HEALPix pixels (ring order) within RMAX of the LMC centre: the cells.
cells = hp.query_disc(NSIDE, hp.ang2vec(LMC['ra0'], LMC['dec0'], lonlat=True), np.radians(RMAX))
count = lambda s: np.bincount(hp.ang2pix(NSIDE, s.ra.to_numpy(), s.dec.to_numpy(), lonlat=True), minlength=hp.nside2npix(NSIDE))[cells]  # noqa: E731
d = pd.DataFrame(dict(pix=cells, n_g=count(p), n_h=count(t)))   # parent and 2MASS counts per cell
d['ra'], d['dec'] = hp.pix2ang(NSIDE, cells, lonlat=True)        # cell centres [deg]
# Distance from the centre [deg] and tangent-plane x (east), y (north) [deg] of each cell.
rho = angular_distance(d.ra.to_numpy(), d.dec.to_numpy(), LMC['ra0'], LMC['dec0'])
x, y = lmc_xy(d.ra.to_numpy(), d.dec.to_numpy())
# Conversion region as in build_giant_mass.py: 4 x ring + (x >= 0) + 2 (y >= 0); x east, y north.
d['region'] = 4 * np.clip(np.digitize(rho, RINGS) - 1, 0, 3) + (x >= 0) + 2 * (y >= 0)
sfd = p.ebv.to_numpy()[cKDTree(xyz(p.ra.to_numpy(), p.dec.to_numpy())).query(xyz(d.ra.to_numpy(), d.dec.to_numpy()))[1]]  # nearest star
ebv = true_ebv(d.ra.to_numpy(), d.dec.to_numpy(), np.full(len(d), np.nan), rc_map())  # NaN outside the RC map
inside = np.isfinite(ebv)   # cells inside the Skowron map (SFD saturates in the LMC, so no fallback)
print(f'{(~inside).sum()} of {len(d)} cells outside the RC map, dropped')
d, sfd, ebv, x, y = d[inside].reset_index(drop=True), sfd[inside], ebv[inside], x[inside], y[inside]
d['evi'] = ebv * EVI_PER_EBV  # back to Skowron E(V-I) [mag]
f = model_ratio()   # interpolator of n_G / n_H on (true E(B-V), carton E(B-V))
# Model ratio at the cell's own true and carton E(B-V), and at the median values of its conversion region.
d['f_cell'] = f(np.column_stack([ebv, magcloud.carton_ebv(d.ra, d.dec, sfd)]))
d['f_region'] = f(np.column_stack([reg.ebv.loc[d.region], reg.ebv_carton.loc[d.region]]))
# One row per cell (columns in the module docstring).
d.to_parquet(DATA / 'parent_extinction_cells.parquet', index=False)
binned_path = ROOT / 'results/parent_extinction.csv'

### Observed over predicted, per E(V-I) bin
Normalised to the cells with E(V-I) < 0.1, with a bootstrap over cells; written to `results/parent_extinction.csv`.

In [ ]:
rng = np.random.default_rng(3)   # fixed seed of the bootstrap over cells
d['bin'] = np.digitize(d.evi, EVI_EDGES) - 1  # E(V-I) bin; len(EVI_EDGES) - 1 for evi >= 0.8 (not reported)


def q(s, col):
    """sum N_G / sum N_H f per E(V-I) bin, normalised to E(V-I) < 0.1.

    Parameters
    ----------
    s : pandas.DataFrame
        Cells with bin, evi, n_g, n_h and f_cell, f_region.
    col : str
        Model: 'cell' or 'region'.

    Returns
    -------
    numpy.ndarray
        q per bin, length len(EVI_EDGES) (the last entry is the overflow bin); NaN where a bin is empty.
    """
    # Observed parent counts and predicted counts N_H f summed per E(V-I) bin (overflow bin included).
    num, den = np.bincount(s.bin, s.n_g, len(EVI_EDGES)), np.bincount(s.bin, s.n_h * s['f_' + col], len(EVI_EDGES))
    low = s.evi < .1   # normalisation cells
    # Ratio per bin divided by the same ratio over the low-reddening cells.
    return num / den / (s.n_g[low].sum() / (s.n_h * s['f_' + col])[low].sum())


rows = []
# For each model: q per bin and the 16th-84th percentiles over the bootstrap draws; the overflow bin (evi >= 0.8) is not reported.
for col in ('cell', 'region'):
    boot = np.array([q(d.iloc[rng.integers(0, len(d), len(d))], col) for _ in range(200)])  # 200 bootstrap draws of cells
    for j, (value, lo, hi) in enumerate(zip(q(d, col), *np.nanpercentile(boot, [16, 84], axis=0))):
        if j < len(EVI_EDGES) - 1:
            rows.append(dict(model=col, evi_lo=EVI_EDGES[j], evi_hi=EVI_EDGES[j + 1], n_cells=int((d.bin == j).sum()),
                             n_g=int(d.n_g[d.bin == j].sum()), q=value, q_p16=lo, q_p84=hi))
# Table per model and E(V-I) bin, written to results/parent_extinction.csv.
binned = pd.DataFrame(rows)
binned.to_csv(binned_path, index=False)
print(binned.round(3).to_string(index=False))
# Per region: median and 99th percentile of cell E(V-I), and the parent fraction the cell-by-cell model expects relative to the
# regional-median model.
per_region = d.groupby('region').apply(lambda s: pd.Series(dict(
    evi_median=s.evi.median(), evi_p99=s.evi.quantile(.99),
    expected_cell_over_region=(s.n_h * s.f_cell).sum() / (s.n_h * s.f_region).sum())), include_groups=False)
print(per_region.round(3).to_string())

### The figure

In [ ]:
plt.style.use(ROOT / 'src/lmcold/paper.mplstyle')
fig, axes = panel_grid(1, 2, width=7, left=.55, right=.1, hgap=.7, bottom=.42, top=.6)
# (a) Cells on the sky coloured by Skowron E(V-I) [mag], 0-0.4; east to the left; colour bar above the panel.
sc = axes[0, 0].scatter(x, y, c=d.evi, s=.6, marker='s', cmap='magma_r', vmin=0, vmax=.4, lw=0, rasterized=True)
axes[0, 0].set(xlim=(RMAX, -RMAX), ylim=(-RMAX, RMAX), xlabel='$x$ (east) [deg]', ylabel='$y$ (north) [deg]')
axes[0, 0].set_title('(a) Skowron+21 $E(V-I)$ in 3.4$^\\prime$ cells', pad=24)
fig.colorbar(sc, cax=axes[0, 0].inset_axes([0, 1.03, 1, .04]), orientation='horizontal').ax.xaxis.set_ticks_position('top')
# (b) q against cell E(V-I) for the two models (offset -+0.004 mag in x), bins with >= 50 parent stars only.
ax = axes[0, 1]
for (col, lab), color, dx in zip([('cell', 'model at the cell $E(V-I)$'), ('region', 'model at the region median')], ['#2878b5', '#d75a20'], [-.004, .004]):
    b = binned[(binned.model == col) & (binned.n_g >= 50)]
    c = (b.evi_lo + b.evi_hi) / 2 + dx
    ax.errorbar(c, b.q, [b.q - b.q_p16, b.q_p84 - b.q], fmt='o-', ms=3, color=color, label=lab)
ax.axhline(1, color='0.3', ls='--', lw=.7)   # q = 1: complete parent with the right dust model
ax.set(xlabel=r'$E(V-I)$ of the cell', ylabel=r'$\sum N_G\,/\,\sum N_H f$, normalised', xlim=(0, .6), ylim=(0, 1.6))
ax.set_title('(b) Gaia parent against 2MASS giants', pad=24)
ax.legend(loc='lower left')
fig.savefig(ROOT / 'figures/paper_parent_extinction.png')
print('Saved paper_parent_extinction.png')

In [ ]:
from IPython.display import Image
Image(filename='figures/paper_parent_extinction.png')

**Figure 4.6.** The *Gaia* parent in the dusty centre of the LMC. (a) Red-clump $E(V-I)$ (mag, colour bar above
the panel) of Skowron et al. (2021) in 3.4-arcmin cells within 4.5° of the centre, against $x$ (east,
increasing to the left) and $y$ (north), in degrees. (b) Parent stars over 2MASS giants (12.35 ≤ $H$ ≤
13.3), divided by the isochrone prediction $f$ and normalised to cells with $E(V-I)$ < 0.10, in bins of
the cell's $E(V-I)$ (mag): $f$ at the reddening of each cell (blue) and at the median of its conversion region, as used for the
masses (orange); dashed: a ratio of 1. Bars: 16th–84th percentiles of a bootstrap over cells; bins with at least 50 parent stars.
`scripts/paper_parent_extinction.py`

**Foreground.** Figure 4.7 shows the foreground fractions of the corrected counts and the test of their assumption.
The carton's proper-motion window keeps only foreground stars with LMC-like motions, so the proper-motion sideband tests the radial
shape of the foreground, not its amount. At 11–12° the model assigns
80–100 per cent of the giants to the foreground, an extrapolation that the zero and double
amplitudes of the model range bracket.

<details><summary>About <code>scripts/fig_giant_foreground.py</code></summary>

```text
Diagnostic: Milky Way foreground of the BOSS giants against radius (§2.3).

(a) Foreground fraction of the model (lmcold.foreground; data/giant_foreground.parquet from build_giant_mass.py), weighted
mean per 1-deg ring, for three metallicity ranges.
(b) Test of its assumption, a uniform foreground surface density: weighted surface density of BOSS giants whose proper motion
(Magellanic Stream frame) lies more than SIDE mas/yr from the local LMC motion (median of the [Fe/H] > -1 giants in the same
nside-16 pixel). The carton keeps only stars within 1.2 mas/yr of a fixed LMC motion, so these sideband stars are a fixed
fraction of the foreground only if its proper-motion distribution does not change across the field; they measure its radial
shape, not its amount. Poisson errors from the raw counts.
Writes figures/fig_giant_foreground.png and results/giant_foreground_sideband.csv.

Paper figure: appendix (Sect. 2.3), docs/figures.md row figures/fig_giant_foreground.png.

Inputs: the weighted BOSS giants of lmcold.mass_samples.samples (release_boss_weights and the files listed there);
data/giant_foreground.parquet (rho [deg], coverage fill, model foreground fraction f_mw; scripts/build_giant_mass.py);
data/boss_parent.parquet (release_boss_parent: Gaia DR3 pmra, pmdec).

Figure (two equal square panels):
(a) x: angular distance rho from the LMC centre [deg], 1-deg rings (lmcold.foreground.RINGS) plotted at their centres; y (log):
    model foreground fraction, the mean of f_mw weighted by w x fill. Orange: [Fe/H] < -1.5; green: -1.5 to -1; blue: > -1.
(b) x: rho [deg] in the rings EDGES (horizontal bars: ring width); y: surface density of sideband giants with [Fe/H] < -1.5,
    sum of w x fill per deg^2, with Poisson errors (density / sqrt(raw count)). Dashed line: the inverse-variance weighted mean
    (the uniform-density hypothesis); its chi2 is printed.

results/giant_foreground_sideband.csv: rho_lo, rho_hi [deg], n_raw (sideband giants), density [deg^-2], err [deg^-2].

Run: .venv/bin/python scripts/fig_giant_foreground.py
```
</details>

**Data read by `scripts/fig_giant_foreground.py`** (from the run log)

| File | Where it comes from | What it is |
|---|---|---|
| `data/apogee_weights.parquet` | Zenodo bundle 1 | APOGEE selection weights per candidate star (selection groups, coverage, w = N_cand / N_quality x pass factor in H x XP [M/H]) |
| `data/boss_parent.parquet` | Zenodo bundle 1 | Gaia DR3 stars within 12 deg of the LMC passing the rough cuts of mwm_magcloud_rgb_boss, with SFD E(B-V) and `eligible` |
| `data/boss_weights.parquet` | Zenodo bundle 1 | BOSS selection weights per eligible parent star (selection groups, w = N_elig / N_good x pass factor in G x XP [M/H] x ring) |
| `data/giant_foreground.parquet` | Zenodo bundle 1 | Milky Way foreground model per giant, from the mass analysis |
| `data/spec_giants.parquet` | Zenodo bundle 1 | APOGEE DR17 and SDSS-V DR20 BOSS spectra of LMC giant candidates before the Gaia match |
| `results/boss_calibration.json` | repository | calibration of the BOSS [Fe/H] and velocities to APOGEE on the stars observed by both; written by `python scripts/build_giants.py` |

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

In [ ]:
import healpy as hp
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from lmcold import magcloud
from lmcold.foreground import RINGS
from lmcold.layout import panel_grid
from lmcold.mass_samples import samples
from lmcold.sources import DATA, ROOT

### Settings

In [ ]:
SIDE = 0.8  # mas/yr: minimum distance from the local LMC proper motion (MS frame) for a sideband star
EDGES = [0, 3, 6, 9, 12]  # rings of panel (b) [deg]

### Load the data
The BOSS mass sample through `lmcold.mass_samples.samples()` (built from the catalogues of Zenodo bundle 1), joined on Gaia
source_id with the foreground fraction of each giant (`data/giant_foreground.parquet`, written by `scripts/build_giant_mass.py`)
and with its Gaia proper motion (`data/boss_parent.parquet`, Zenodo bundle 1).

In [ ]:
# lmcold.mass_samples.samples(): BOSS giants with calibrated [Fe/H] and selection weight w
b, _, _ = samples()
# data/giant_foreground.parquet: rho [deg], coverage fill and Milky Way foreground fraction f_mw of each BOSS giant
b = b.merge(pd.read_parquet(DATA / "giant_foreground.parquet")[["source_id", "rho", "fill", "f_mw"]], on="source_id")
# data/boss_parent.parquet: Gaia proper motions pmra, pmdec [mas/yr]
b = b.merge(pd.read_parquet(DATA / "boss_parent.parquet", columns=["source_id", "pmra", "pmdec"]), on="source_id")

### The sideband sample
BOSS giants with their model foreground fraction, and their proper motion relative to the local LMC motion.

In [ ]:
# Proper motions in the Magellanic Stream frame (Nidever et al. 2008): pm_L cos B, pm_B [mas/yr].
_, _, b["pl"], b["pb"] = magcloud.ms_coordinates(b.ra.to_numpy(), b.dec.to_numpy(), b.pmra.to_numpy(), b.pmdec.to_numpy())
# Local LMC motion: median MS-frame proper motion of the [Fe/H] > -1 giants in the same HEALPix nside-16 (NESTED, ~3.7 deg)
# pixel; pixels without such giants give NaN (those stars are dropped from panel b).
b["pix"] = hp.ang2pix(16, b.ra.to_numpy(), b.dec.to_numpy(), lonlat=True, nest=True)
b = b.join(b[b.feh > -1].groupby("pix")[["pl", "pb"]].median(), on="pix", rsuffix="_lmc")
b["side"] = np.hypot(b.pl - b.pl_lmc, b.pb - b.pb_lmc) > SIDE

### Panel (a): model fraction per ring; panel (b): sideband density
The sideband densities go to `results/giant_foreground_sideband.csv`.

In [ ]:
plt.style.use(ROOT / "src/lmcold/paper.mplstyle")
fig, ax = panel_grid(1, 2, width=5.0, left=0.6, right=0.1, hgap=0.7, bottom=0.45, top=0.55)
cols = {"[Fe/H] < −1.5": ("#d75a20", b.feh < -1.5), "−1.5 to −1": ("#33976b", (b.feh >= -1.5) & (b.feh < -1)),
        "> −1": ("#2878b5", b.feh >= -1)}
centre = 0.5 * (RINGS[1:] + RINGS[:-1])
for lab, (col, m) in cols.items():
    # Weighted mean of the per-star model fraction f_mw in each ring, weights w x fill.
    k = np.digitize(b.rho[m], RINGS) - 1
    w = (b.w * b["fill"])[m].to_numpy()
    ax[0, 0].plot(centre, np.bincount(k, w * b.f_mw[m], len(centre)) / np.bincount(k, w, len(centre)), "o-", ms=3, lw=0.8,
                  color=col, label=lab)
ax[0, 0].set(yscale="log", ylim=(1e-3, 1), xlim=(0, 12), xlabel=r"$\rho$ [deg]", ylabel="foreground fraction (model)", title="(a) Model")
ax[0, 0].legend(loc="lower left", bbox_to_anchor=(0, 1.12), borderaxespad=0, ncol=3)
# Solid angle of each ring [deg^2]: 2 pi (cos rho_in - cos rho_out) sr x (180/pi)^2.
area = 2 * np.pi * -np.diff(np.cos(np.radians(EDGES))) * (180 / np.pi) ** 2
mp = b[(b.feh < -1.5) & b.side & b.pl_lmc.notna()]
k = np.digitize(mp.rho, EDGES) - 1
dens = np.bincount(k, (mp.w * mp["fill"]).to_numpy(), len(area)) / area
n = np.bincount(k, minlength=len(area))
err = dens / np.sqrt(np.maximum(n, 1))
c = 0.5 * (np.array(EDGES[1:]) + np.array(EDGES[:-1]))
ax[0, 1].errorbar(c, dens, yerr=err, xerr=1.5, fmt="o", ms=3, color="#d75a20", lw=0.8)
mean = np.sum(dens / err ** 2) / np.sum(1 / err ** 2)
ax[0, 1].axhline(mean, color="0.4", ls="--", lw=0.7)
ax[0, 1].set(xlim=(0, 12), ylim=(0, None), xlabel=r"$\rho$ [deg]", ylabel=r"sideband giants per deg$^2$",
             title=r"(b) [Fe/H] $<-1.5$, $|\Delta\mu| > 0.8$ mas yr$^{-1}$")
fig.savefig(ROOT / "figures/fig_giant_foreground.png")
t = pd.DataFrame(dict(rho_lo=EDGES[:-1], rho_hi=EDGES[1:], n_raw=n, density=dens, err=err))
t.to_csv(ROOT / "results/giant_foreground_sideband.csv", index=False)
chi2 = float(np.sum(((dens - mean) / err) ** 2))
print(t.round(3).to_string(index=False), f"\nchi2 about a constant {chi2:.1f} for {len(dens) - 1} dof")

In [ ]:
from IPython.display import Image
Image(filename='figures/fig_giant_foreground.png')

**Figure 4.7.** Milky Way foreground of the BOSS giants. (a) Model foreground fraction (logarithmic) per ring of 1°
in angular distance $\rho$ (deg), for three [Fe/H] ranges: below −1.5 (orange), −1.5 to −1 (green) and above −1 (blue). (b) Surface
density (giants per deg²) of the giants with [Fe/H] < −1.5 whose proper motion lies more than 0.8 mas/yr from the
local LMC motion, in rings of $\rho$ (horizontal bars: ring width; vertical bars: Poisson errors); the dashed line is the weighted
mean. `scripts/fig_giant_foreground.py`